In [ ]:
# import shutil

# # Replace 'your_folder_name' with your actual folder path
# folder_path = '/content/LECO'

# shutil.rmtree(folder_path)

01. ENVIORNMENT SETUP and DATA LOADING

In [ ]:
import os, shutil

DRIVE_ROOT = "/content/drive/MyDrive/LECO"
DEST = "/content/LECO"

# Flat subfolders: just the folder name
SUBFOLDERS = [
    "concepts_files_output",
    "nodes_files output",
    "options_files_output",
    "error_files_output"
]

# Nested subfolders: (relative/path/to/folder, local_destination_name)
NESTED = [
    ("PYQ raw_ques/math_raw_json", "PYQ_raw"),
]

# NEW: Individual files located directly in the LECO folder
# Just write the file name and extension
FILES = [
    "unsolved.json",   # Replace with your actual JSON file name
    "df_exemplerss.csv"     # Replace with your actual CSV file name
]

# Function to copy entire folders
def sync_folder(src, dst):
    if not os.path.exists(src): print(f"Folder not found: {src}"); return
    if os.path.exists(dst): shutil.rmtree(dst)
    shutil.copytree(src, dst)

# NEW: Function to copy individual files
def sync_file(src_file, dst_dir):
    if not os.path.exists(src_file): print(f"File not found: {src_file}"); return
    if not os.path.exists(dst_dir): os.makedirs(dst_dir) # Ensure LECO folder exists locally
    shutil.copy2(src_file, dst_dir)

# 1. Sync Flat Folders
for folder in SUBFOLDERS:
    sync_folder(os.path.join(DRIVE_ROOT, folder), os.path.join(DEST, folder))

# 2. Sync Nested Folders
for path, name in NESTED:
    sync_folder(os.path.join(DRIVE_ROOT, path), os.path.join(DEST, name))

# 3. Sync Individual Files
for file_name in FILES:
    sync_file(os.path.join(DRIVE_ROOT, file_name), DEST)

# Print Summary for Folders
for folder in SUBFOLDERS + [name for _, name in NESTED]:
    if os.path.exists(os.path.join(DEST, folder)):
        count = sum(len(files) for _, _, files in os.walk(os.path.join(DEST, folder)))
        print(f"Folder '{folder}': {count} files")

# Print Summary for Files
print("---")
for file_name in FILES:
    if os.path.exists(os.path.join(DEST, file_name)):
        print(f"File copied successfully: {file_name}")

Folder 'concepts_files_output': 57 files
Folder 'nodes_files output': 57 files
Folder 'options_files_output': 149 files
Folder 'error_files_output': 73 files
Folder 'PYQ_raw': 27 files
---
File copied successfully: unsolved.json
File copied successfully: df_exemplerss.csv


In [ ]:
import json
import glob
import pandas as pd



FOLDER_PATH = "/content/LECO/PYQ_raw/*.json"  # UPDATE: folder with raw question JSONs only



file_list = glob.glob(FOLDER_PATH)
print(f"Found {len(file_list)} question files")

all_questions = []
meta_rows = []

for file_path in sorted(file_list):
    with open(file_path, 'r') as f:
        data = json.load(f)

    meta = data.get('meta', {})

    # Clean topic name
    topic = meta.get('topic', '').replace('-', ' ').title()

    # Clean subject
    raw_subject = str(meta.get('subject', '')).strip()
    if raw_subject in ['chem', 'chemistry']:
        subject = 'Chemistry'
    elif raw_subject in ['math', 'maths', 'Mathematics']:
        subject = 'Mathematics'
    elif raw_subject in ['phy', 'physics']:
        subject = 'Physics'
    else:
        subject = 'Unknown'

    # --- A. Extract Questions ---
    for q in data.get('questions', []):
        all_questions.append({
            'raw_id':        q.get('id'),
            'topic':         topic,
            'subject':       subject,
            'year':          q.get('year'),
            'shift':         q.get('shift'),
            'question_type': q.get('question_type'),
            'question_text': q.get('question_text'),
        })

    # --- B. Extract Metadata ---
    meta_row = {
        'topic':      topic,
        'subject':    subject,
        'total':      meta.get('total', 0),
        'mcq':        meta.get('mcq', 0),
        'numerical':  meta.get('numerical', 0),
    }

    # Add year columns dynamically
    year_dist = meta.get('year_distribution', {})
    for year, count in sorted(year_dist.items()):
        meta_row[str(year)] = count

    meta_rows.append(meta_row)



# 1. Questions DataFrame
df_pyq_raw = pd.DataFrame(all_questions)

# 2. Meta DataFrame
df_pyq_meta = pd.DataFrame(meta_rows)

if not df_pyq_meta.empty:
    # Fill missing years with 0 and format
    year_cols = sorted([c for c in df_pyq_meta.columns if c.isdigit()])
    df_pyq_meta[year_cols] = df_pyq_meta[year_cols].fillna(0).astype(int)
    # Sort alphabetically
    df_pyq_meta = df_pyq_meta.sort_values('topic').reset_index(drop=True)


df_pyq_raw
# df_pyq_meta

Found 27 question files


,raw_id,topic,subject,year,shift,question_type,question_text
0,1,3D Geometry,Mathematics,2026,28th January Morning,MCQ,"If the distances of the point (1, 2, a) from t..."
1,2,3D Geometry,Mathematics,2026,24th January Evening,MCQ,"The sum of all values of α, for which the shor..."
2,3,3D Geometry,Mathematics,2026,23rd January Morning,MCQ,Let the direction cosines of two lines satisfy...
3,4,3D Geometry,Mathematics,2026,23rd January Morning,MCQ,The vertices B and C of a triangle ABC lie on ...
4,5,3D Geometry,Mathematics,2026,22nd January Evening,MCQ,Let L be the line (x + 1)/2 = (y + 1)/3 = (z +...
...,...,...,...,...,...,...,...
4476,193,Vector Algebra,Mathematics,2020,4th September Evening,Numerical,"If a⃗ = 2î + ĵ + 2k̂, then the value of |î × (..."
4477,194,Vector Algebra,Mathematics,2020,2nd September Evening,Numerical,Let the position vectors of points 'A' and 'B'...
4478,195,Vector Algebra,Mathematics,2020,2nd September Morning,Numerical,"Let a⃗, b⃗ and c⃗ be three unit vectors such t..."
4479,196,Vector Algebra,Mathematics,2020,9th January Evening,Numerical,"Let a⃗, b⃗ and c⃗ be three vectors such that |..."


In [ ]:
# Define Maps For SMART IDS

TOPIC_INDEX_MAP = {
    '3D Geometry':                               '01',
    'Application Of Derivatives':                '02',
    'Area Under The Curves':                     '03',
    'Binomial Theorem':                          '04',
    'Circle':                                    '05',
    'Complex Numbers':                           '06',
    'Definite Integration':                      '07',
    'Differentiation':                           '08',
    'Ellipse':                                   '09',
    'Functions':                                 '10',
    'Hyperbola':                                 '11',
    'Indefinite Integrals':                      '12',
    'Inverse Trigonometric Functions':           '13',
    'Limits Continuity And Differentiability':   '14',
    'Logarithm':                                 '15',
    'Matrices And Determinants':                 '16',
    'Parabola':                                  '17',
    'Permutations And Combinations':             '18',
    'Probability':                               '19',
    'Quadratic Equation And Inequalities':       '20',
    'Sequences And Series':                      '21',
    'Sets And Relations':                        '22',
    'Statistics':                                '23',
    'Straight Lines And Pair Of Straight Lines': '24',
    'Trigonometric Ratio And Identites':         '25',
    'Vector Algebra':                            '26',
}


SUBJECT_CODE_MAP = {
    'Chemistry':    'C',
    'Mathematics':  'M',
    'Physics':      'P',
}

if not df_pyq_raw.empty:
    # 1. Map Subject Code (Default to 'X' if not found)
    subj_code = df_pyq_raw['subject'].map(SUBJECT_CODE_MAP).fillna('X')

    # 2. Map Topic Index (Default to '00' if not found)
    top_idx = df_pyq_raw['topic'].map(TOPIC_INDEX_MAP).fillna('00')

    # 3. Format raw ID to 3 digits (e.g., "1" -> "001")
    padded_id = df_pyq_raw['raw_id'].astype(str).str.zfill(3)

    # 4. Construct final Question ID string
    df_pyq_raw.insert(0, 'question_id', 'Q_' + subj_code + '_' + top_idx + '_' + padded_id)

    # 5. Drop the temporary raw_id column
    df_pyq_raw = df_pyq_raw.drop(columns=['raw_id'])



In [ ]:
# ==========================================
# STEP 1: SETUP CONTAINERS
# ==========================================
# We create an empty list for each of our target tables (and one temporary for demands).
list_nodes = []
list_demands = []
list_exemplars = []
list_concepts = []
list_errors = []
list_dimensions = []

# Path to your JSON files
folder_path = "/content/PYQ_analysis/*.json"
file_list = glob.glob(folder_path)

# ==========================================
# STEP 2: LOOP & EXTRACT (File by File)
# ==========================================
for file_path in file_list:
    with open(file_path, 'r') as f:
        data = json.load(f)

    # Clean the overarching metadata once per file
    topic = data.get('topic', '').replace('-', ' ').title()




    raw_subject = str(data.get('subject', '')).strip()
    if raw_subject in ['chem', 'chemistry','Chemistry']:
        subject = 'Chemistry'
    elif raw_subject in ['math', 'maths', 'Mathematics']:
        subject = 'Mathematics'
    elif raw_subject in ['phy', 'physics']:
        subject = 'Physics'
    else:
        subject = 'Unknown'

    # 1. Archetypes (Base for dna_nodes)
    if 'archetypes' in data:
        df_a = pd.json_normalize(data, record_path=['archetypes'])
        df_a['source_topic'] = topic
        df_a['subject'] = subject
        list_nodes.append(df_a)

        # 2. Exemplars (Nested inside archetypes)
        df_e = pd.json_normalize(
            data,
            record_path=['archetypes', 'exemplars'],
            meta=[['archetypes', 'archetype_name']],
            errors='ignore'
        )
        df_e['source_topic'] = topic
        df_e['subject'] = subject
        list_exemplars.append(df_e)

    # 3. Cognitive Demands (Temporary table to merge into dna_nodes)
    if 'cognitive_demand' in data:
        df_cd = pd.json_normalize(data, record_path=['cognitive_demand'])
        df_cd['source_topic'] = topic
        list_demands.append(df_cd)

    # 4. Concept Inventory
    # (Handling as a flat list of strings based on your previous Colab setup)
    concepts = data.get('concept_inventory', [])
    if concepts:
        df_c = pd.DataFrame({'concept_name': concepts})
        df_c['topic'] = topic
        df_c['subject'] = subject
        list_concepts.append(df_c)

    # 5. Error Taxonomy
    if 'error_taxonomy' in data:
        df_err = pd.json_normalize(data, record_path=['error_taxonomy'])
        df_err['topic'] = topic
        df_err['subject'] = subject
        list_errors.append(df_err)

    # 6. Classification Dimensions
    if 'classification_dimensions' in data:
        df_dim = pd.json_normalize(data, record_path=['classification_dimensions'])
        df_dim['topic'] = topic
        df_dim['subject'] = subject
        list_dimensions.append(df_dim)

# ==========================================
# STEP 3: CONSOLIDATE
# ==========================================
# Combine all lists into master dataframes
df_base_nodes = pd.concat(list_nodes, ignore_index=True)
df_all_demands = pd.concat(list_demands, ignore_index=True)
df_exemplars = pd.concat(list_exemplars, ignore_index=True)
df_concepts_q = pd.concat(list_concepts, ignore_index=True) if list_concepts else pd.DataFrame()
df_error_tax = pd.concat(list_errors, ignore_index=True)
df_dimensions = pd.concat(list_dimensions, ignore_index=True)

# ==========================================
# STEP 4: MASTER TOPIC INDEXING (NEW)
# ==========================================
# Grab all unique topics, sort them ignoring case, and map them to "01", "02", etc.
all_topics = pd.concat([df_base_nodes['source_topic'], df_error_tax['topic'], df_dimensions['topic']]).dropna().unique()
sorted_topics = sorted(all_topics, key=lambda x: str(x).lower())
topic_index_map = {topic: f"{i+1:02d}" for i, topic in enumerate(sorted_topics)}


def apply_smart_ids(df, topic_col, prefix, id_col_name):
    if df.empty: return df

    # 1. Sort Alphabetically by Topic
    df = df.sort_values(by=topic_col, key=lambda x: x.str.lower()).reset_index(drop=True)

    # 2. Map the Topic Index (01, 02...)
    df['t_idx'] = df[topic_col].map(topic_index_map)

    # 3. Create a local counter per topic (001, 002, 003...)
    df['local_count'] = df.groupby(topic_col).cumcount() + 1
    df['local_idx'] = df['local_count'].astype(str).str.zfill(3)

    # 4. Map the Subject to a single letter code (C, M, P)
    subj_code_map = {'Chemistry': 'C', 'Mathematics': 'M', 'Physics': 'P'}
    df['s_code'] = df['subject'].map(subj_code_map).fillna('X')

    # 5. Construct the final ID string (e.g., N_M_01_001 or N_C_05_012)
    df[id_col_name] = prefix + "_" + df['s_code'] + "_" + df['t_idx'] + "_" + df['local_idx']

    # Drop temp columns
    return df.drop(columns=['t_idx', 'local_count', 'local_idx', 's_code'])

# ==========================================
# STEP 5: CLEAN, MERGE & FORMAT TO SCHEMA
# ==========================================

# --- TABLE 1: DNA NODES ---
# Merge base nodes with cognitive demands
df_dna_nodes = pd.merge(
    df_base_nodes,
    df_all_demands[['source_topic', 'archetype_name', 'position','reasoning']],
    on=['source_topic', 'archetype_name'],
    how='left'
)

# Rename and Map Cognitive Demand
df_dna_nodes = df_dna_nodes.rename(columns={
    'archetype_name': 'node_name',
    'position': 'cognitive_demand'
})


cog_map = {
    # Version 1 — old human readable
    "Retrieval + substitution": "Direct Application",
    "retrieval + substitution": "Direct Application",
    "Setup-heavy": "Build Then Solve",
    "setup-heavy": "Build Then Solve",
    "Inversion/synthesis": "Reverse Engineer",
    "Inversion / synthesis": "Reverse Engineer",
    "inversion / synthesis": "Reverse Engineer",

    # Version 2 — snake_case
    "retrieval_substitution": "Direct Application",
    "retrieval_to_setup": "Formula with Judgment",
    "setup_heavy": "Build Then Solve",
    "setup_to_inversion": "Build and Work Backwards",
    "inversion_synthesis": "Reverse Engineer",

    # Span values — Version 1
    "Retrieval + substitution to Setup-heavy": "Direct Application to Build Then Solve",
    "Setup-heavy to Inversion/synthesis": "Build Then Solve to Reverse Engineer",
    "Setup-heavy to Inversion / synthesis": "Build Then Solve to Reverse Engineer",

    # Span values — Version 2
    "retrieval_substitution to retrieval_to_setup": "Direct Application to Formula with Judgment",
    "retrieval_to_setup to setup_heavy": "Formula with Judgment to Build Then Solve",
    "retrieval_to_setup to setup_to_inversion": "Formula with Judgment to Build and Work Backwards",
    "retrieval_substitution to setup_heavy": "Direct Application to Build Then Solve",
    "setup_heavy to setup_to_inversion": "Build Then Solve to Build and Work Backwards",
    "setup_to_inversion to inversion_synthesis": "Build and Work Backwards to Reverse Engineer",}



df_dna_nodes['cognitive_demand'] = df_dna_nodes['cognitive_demand'].map(cog_map).fillna(df_dna_nodes['cognitive_demand'])


# Generate Smart IDs: N_C_01_001
df_dna_nodes = apply_smart_ids(df_dna_nodes, 'source_topic', 'N', 'node_id')
df_dna_nodes = df_dna_nodes[['node_id', 'node_name', 'cognitive_operation', 'subject', 'source_topic', 'reasoning', 'cognitive_demand']]


# --- TABLE 2: EXEMPLARS ---
df_exemplars = df_exemplars.rename(columns={'archetypes.archetype_name': 'node_name', 'excerpt': 'excerpt_text'})

# Bring in the newly generated node_id
df_exemplars = pd.merge(
    df_exemplars, df_dna_nodes[['node_name', 'source_topic', 'node_id']],
    on=['node_name', 'source_topic'], how='left'
)


# Generate Smart IDs: EX_C_01_001
df_exemplars = apply_smart_ids(df_exemplars, 'source_topic', 'EX', 'exemplar_id')
df_exemplars = df_exemplars[['exemplar_id', 'node_id', 'excerpt_text', 'year','source_topic']]


# --- TABLE 3: CONCEPT INVENTORY ---
if not df_concepts_q.empty:
    df_concepts_q['source'] = 'pyq'
    # Generate Smart IDs: CON_C_01_001
    df_concepts_q = apply_smart_ids(df_concepts_q, 'topic', 'CON', 'concept_id')
    df_concepts_q = df_concepts_q[['concept_id', 'topic', 'subject', 'concept_name', 'source']]


# --- TABLE 4: ERROR TAXONOMY ---
# Generate Smart IDs: ERR_C_01_001
df_error_tax = apply_smart_ids(df_error_tax, 'topic', 'ERR', 'error_id')
df_error_tax = df_error_tax[['error_id', 'topic', 'subject', 'error_name', 'description', 'error_type']]


# --- TABLE 5: QUESTION DIMENSIONS ---
# Generate Smart IDs: DIM_C_01_001
df_dimensions = apply_smart_ids(df_dimensions, 'topic', 'DIM', 'dimension_id')
df_dimensions = df_dimensions[['dimension_id', 'topic', 'subject', 'dimension_name', 'low_end', 'high_end', 'diagnostic_value']]


print(f"✅ DNA Nodes Table: {len(df_dna_nodes)} rows")
print(f"✅ Exemplars Table: {len(df_exemplars)} rows")
print(f"✅ Concepts Table: {len(df_concepts_q)} rows")
print(f"✅ Error Taxonomy Table: {len(df_error_tax)} rows")
print(f"✅ Dimensions Table: {len(df_dimensions)} rows")

✅ DNA Nodes Table: 363 rows
✅ Exemplars Table: 1055 rows
✅ Concepts Table: 1048 rows
✅ Error Taxonomy Table: 286 rows
✅ Dimensions Table: 138 rows




---



02. Exemplar Matching & Validation

In [ ]:
!pip install thefuzz[speedup]

import pandas as pd
import re
from thefuzz import fuzz
from thefuzz import process


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 69.9 MB/s eta 0:00:00


In [ ]:
df_exemplars = pd.read_csv("/content/LECO/df_exemplerss.csv")

In [ ]:
# Create temporary exact match results
df_pyq_exact = df_pyq_raw[['question_id', 'question_text', 'topic']].copy()
df_pyq_exact['is_exact'] = False
df_pyq_exact['exact_id'] = None

df_pyq_exact['clean_q'] = df_pyq_exact['question_text'].str.strip().str.lower()
for _, row in df_exemplars.iterrows():
    search = str(row['excerpt_text']).replace('...', '').strip().lower()
    mask = (df_pyq_exact['topic'] == row['source_topic']) & (df_pyq_exact['clean_q'] == search)
    if mask.any():
        df_pyq_exact.loc[mask, ['is_exact', 'exact_id']] = [True, row['exemplar_id']]

print(f"Exact Method found: {df_pyq_exact['is_exact'].sum()} matches")

Exact Method found: 608 matches


In [ ]:
# Create temporary fuzzy match results
df_pyq_fuzzy = df_pyq_raw[['question_id', 'question_text', 'topic']].copy()
df_pyq_fuzzy['is_fuzzy'] = False
df_pyq_fuzzy['fuzzy_id'] = None


def clean_for_match(text):
    if pd.isna(text):
        return ""
    text = str(text).lower()
    text = re.sub(r'[^a-z0-9]', '', text) # Remove all non-alphanumeric chars
    return text


# 2. PREPARE THE DATA
# Create cleaned text columns for matching
df_pyq_fuzzy['match_text'] = df_pyq_fuzzy['question_text'].apply(clean_for_match)
df_exemplars['match_text'] = df_exemplars['excerpt_text'].apply(clean_for_match)

# Prepare dictionary to store matches: { 'question_id' : ('exemplar_id', 'node_id') }
match_dict = {}
unmatched_exemplars = []




# 3. EXECUTE TOPIC-BY-TOPIC FUZZY MATCHING
topics = df_exemplars['source_topic'].unique()

for topic in topics:
    ex_subset = df_exemplars[df_exemplars['source_topic'] == topic]
    pyq_subset = df_pyq_fuzzy[df_pyq_fuzzy['topic'] == topic]

    if pyq_subset.empty:
        continue

    pyq_text_to_id = dict(zip(pyq_subset['match_text'], pyq_subset['question_id']))
    pyq_choices = list(pyq_text_to_id.keys())

    for _, ex_row in ex_subset.iterrows():
        ex_text = ex_row['match_text']
        best_match = process.extractOne(ex_text, pyq_choices, scorer=fuzz.ratio)

        if best_match:
            matched_string, score = best_match
            if score >= 90:
                q_id = pyq_text_to_id[matched_string]

                # --- THIS IS THE FIX: Move the update inside the loop ---
                df_pyq_fuzzy.loc[df_pyq_fuzzy['question_id'] == q_id, ['is_fuzzy', 'fuzzy_id']] = [True, ex_row['exemplar_id']]
                # --------------------------------------------------------

            else:
                unmatched_exemplars.append((ex_row['exemplar_id'], score))
        else:
            unmatched_exemplars.append((ex_row['exemplar_id'], 0))

print(f"Fuzzy Matching complete! Found {df_pyq_fuzzy['is_fuzzy'].sum()} matches.")

Fuzzy Matching complete! Found 921 matches.


In [ ]:
# Combine into a comparison table
df_comp = pd.merge(df_pyq_exact[['question_id', 'exact_id']],
                   df_pyq_fuzzy[['question_id', 'fuzzy_id']], on='question_id')


df_comp['final_id'] = df_comp['fuzzy_id'].combine_first(df_comp['exact_id'])
both = df_comp.dropna(subset=['fuzzy_id', 'exact_id'])
fuzzy_only = df_comp[df_comp['fuzzy_id'].notna() & df_comp['exact_id'].isna()]


# Total Unique Questions Matched (This is your 605 count)
print(f"Total Unique Questions Matched: {df_comp['final_id'].notna().sum()}")

print(f"Matched by BOTH methods:: {len(df_comp[df_comp['exact_id'].notna() & df_comp['fuzzy_id'].notna()])}")
print(f"Fuzzy Only: {len(df_comp[df_comp['exact_id'].isna() & df_comp['fuzzy_id'].notna()])}")
print(f"Exact Only: {len(df_comp[df_comp['fuzzy_id'].isna() & df_comp['exact_id'].notna()])}")

Total Unique Questions Matched: 921
Matched by BOTH methods:: 608
Fuzzy Only: 313
Exact Only: 0


In [ ]:
# Inspect Fuzzy Only matches side-by-side
fuzzy_only_ids = df_comp[df_comp['exact_id'].isna() & df_comp['fuzzy_id'].notna()]['question_id']
view = df_pyq_raw[df_pyq_raw['question_id'].isin(fuzzy_only_ids)][['question_id', 'question_text']].copy()
view['exemplar_text'] = view['question_id'].map(df_pyq_fuzzy.set_index('question_id')['fuzzy_id']).map(df_exemplars.set_index('exemplar_id')['excerpt_text'])
display(view)

,question_id,question_text,exemplar_text
9,Q_M_01_010,Let the values of λ for which the shortest dis...,Let the values of λ for which the shortest dis...
15,Q_M_01_016,"Let the values of p, for which the shortest di...","Let the values of p, for which the shortest di..."
20,Q_M_01_021,"Let a line passing through the point (4, 1, 0)...","Let a line passing through the point (4, 1, 0)..."
58,Q_M_01_059,"Let (α, β, γ) be the mirror image of the point...","Let (α, β, γ) be the mirror image of the point..."
82,Q_M_01_083,If equation of the plane that contains the poi...,If equation of the plane that contains the poi...
...,...,...,...
4408,Q_M_26_125,"Let a⃗, b⃗ and c⃗ be three unit vectors such t...","Let a⃗, b⃗ and c⃗ be three unit vectors such t..."
4409,Q_M_26_126,"Let a⃗, b⃗ and c⃗ be three non-zero vectors su...","Let a⃗, b⃗ and c⃗ be three non-zero vectors su..."
4415,Q_M_26_132,"If u⃗, v⃗ and w⃗ are three non-coplanar vector...","If u⃗, v⃗ and w⃗ are three non-coplanar vector..."
4451,Q_M_26_168,Let a⃗ and b⃗ be two vectors such that |a⃗ + b...,Let a⃗ and b⃗ be two vectors such that |a⃗ + b...


In [ ]:
# 1. Map the fuzzy/both IDs into the main dataframe
df_pyq_raw['is_exemplar'] = False
df_pyq_raw['exemplar_id'] = None

# Create a mapping series from our comparison table
valid_ids = pd.concat([both['question_id'], fuzzy_only['question_id']]).unique()
id_mapping = df_comp[df_comp['question_id'].isin(valid_ids)].set_index('question_id')['fuzzy_id']

# 2. Update the main table
df_pyq_raw.loc[df_pyq_raw['question_id'].isin(id_mapping.index), 'is_exemplar'] = True
df_pyq_raw['exemplar_id'] = df_pyq_raw['question_id'].map(id_mapping)


# 3. Final Check
print(f"Total count of True exemplars: {df_pyq_raw['is_exemplar'].sum()}")

# View a sample
display(df_pyq_raw[df_pyq_raw['is_exemplar'] == True].sample(5))

Total count of True exemplars: 921


,question_id,topic,subject,year,shift,question_type,question_text,is_exemplar,exemplar_id
4175,Q_M_24_115,Straight Lines And Pair Of Straight Lines,Mathematics,2016,9th April Morning Slot,MCQ,"The point (2, 1) is translated parallel to the...",True,EX_M_24_033
461,Q_M_02_127,Application Of Derivatives,Mathematics,2019,8th April Evening Slot,MCQ,The height of a right circular cylinder of max...,True,EX_M_02_036
1770,Q_M_08_029,Differentiation,Mathematics,2022,24th June Evening,MCQ,"If y = tan⁻¹(sec(x/3) − tan(x/3)), π/2 < x/3 <...",True,EX_M_08_029
3411,Q_M_20_009,Quadratic Equation And Inequalities,Mathematics,2026,21st January Evening,MCQ,Let α and β be the roots of the equation x² + ...,True,EX_M_20_008
1065,Q_M_05_138,Circle,Mathematics,2021,31st August Evening,Numerical,Let B be the centre of the circle x² + y² - 2x...,True,EX_M_05_033


In [ ]:
# 1. Identify Orphan IDs (The ones currently NOT marked as exemplars in your DF)
# We look for exemplar_ids that are in df_exemplars but not currently assigned in df_pyq_raw
matched_exemplar_ids = df_pyq_raw[df_pyq_raw['is_exemplar'] == True]['exemplar_id'].unique()
unmarked_exemplars_df = df_exemplars[~df_exemplars['exemplar_id'].isin(matched_exemplar_ids)]

print(f"Attempting to rescue {len(unmarked_exemplars_df)}")

# 2. Rescue Loop: Update df_pyq_raw directly
rescued_count = 0
rescue_data = [] # For verification later

for _, row in unmarked_exemplars_df.iterrows():
    target_excerpt = str(row['excerpt_text']).strip()
    topic = row['source_topic']

    # 50-character fingerprint match
    fingerprint = target_excerpt[:50]

    mask = (df_pyq_raw['topic'] == topic) & \
           (df_pyq_raw['question_text'].str.contains(fingerprint, case=False, regex=False, na=False))

    # Only match if exactly one question is found to avoid duplicates
    if mask.sum() == 1:
        idx = df_pyq_raw[mask].index[0]

        # Directly update the master dataframe
        df_pyq_raw.at[idx, 'is_exemplar'] = True
        df_pyq_raw.at[idx, 'exemplar_id'] = row['exemplar_id']
        df_pyq_raw.at[idx, 'node_id'] = row['node_id']

        # Save for verification
        rescue_data.append({
            'question_text': df_pyq_raw.at[idx, 'question_text'],
            'excerpt_text': row['excerpt_text']
        })
        rescued_count += 1

print(f"Rescued {rescued_count} orphans into df_pyq_raw.")

# 3. Final Verification (Side-by-side)
df_rescue_verification = pd.DataFrame(rescue_data)
pd.set_option('display.max_colwidth', None)
print("\n--- Verification: Side-by-Side Rescue Matches ---")
display(df_rescue_verification.head(10))
pd.reset_option('display.max_colwidth')

Attempting to rescue 104
Rescued 78 orphans into df_pyq_raw.

--- Verification: Side-by-Side Rescue Matches ---


,question_text,excerpt_text
0,"Let the equation of plane passing through the line of intersection of the planes x + 2y + az = 2 and x - y + z = 3 be 5x - 11y + bz = 6a - 1. For c ∈ ℤ, if the distance of this plane from the point (a, -c, c) is 2a, then a + bc is equal to:",Let the equation of plane passing through the line of intersection of the planes x + 2y + az = 2 and x - y + z = 3 be 5x - 11y + bz = 6a - 1. For c ∈ ℤ...
1,"Let the plane 2x + 3y + z + 20 = 0 be rotated through a right angle about its line of intersection with the plane x − 3y + 5z = 8. If the mirror image of the point (2, −1/2, 2) in the rotated plane is B(a, b, c), then:",Let the plane 2x + 3y + z + 20 = 0 be rotated through a right angle about its line of intersection with the plane x − 3y + 5z = 8.
2,"A plane P contains the line of intersection of the plane r⃗ · (î + ĵ + k̂) = 6 and r⃗ · (2î + 3ĵ + 4k̂) = -5. If P passes through the point (0, 2, -2), then the square of distance of the point (12, 12, 18) from the plane P is:","A plane P contains the line of intersection of the plane r⃗ · (î + ĵ + k̂) = 6 and r⃗ · (2î + 3ĵ + 4k̂) = -5. If P passes through the point (0, 2, -2)"
3,"Let the foot of perpendicular from a point P(1, 2, −1) to the straight line L: x/1 = y/0 = z/(−1) be N. Let a line be drawn from P parallel to the plane x + y + 2z = 0 which meets L at point Q. If α is the acute angle between the lines PN and PQ, then cos α is equal to ____.","Let the foot of perpendicular from a point P(1, 2, −1) to the straight line L: x/1 = y/0 = z/(−1) be N."
4,Let A be the point of intersection of the lines L₁: (x - 7)/1 = (y - 5)/0 = (z - 3)/(-1) and L₂: (x - 1)/3 = (y + 3)/4 = (z + 7)/5. Let B and C be the points on the lines L₁ and L₂ respectively such that AB = AC = 15. Then the square of the area of the triangle ABC is,Let B and C be the points on the lines L₁ and L₂ respectively such that AB = AC = 15. Then the square of the area of the triangle ABC is
5,"Let f : ℝ → ℝ be a twice differentiable function such that f″(x) > 0 for all x ∈ ℝ and f′(a − 1) = 0, where a is a real number. Let g(x) = f(tan²x − 2tanx + a), 0 < x < π/2. Consider the following two statements: (I) g is increasing in (0, π/4) (II) g is decreasing in (π/4, π/2) Then,","Let f : ℝ → ℝ be a twice differentiable function such that f″(x) > 0 for all x ∈ ℝ and f′(a − 1) = 0, where a is a real number. Let g(x) = f(tan²x − 2tanx + a), 0 < x < π/2."
6,"The area of the region {(x, y) : y² ≤ 4x, x < 4, xy(x − 1)(x − 2)(x − 3)(x − 4) > 0, x ≠ 3} is","The area of the region {(x, y) : y² ≤ 4x, x < 4, xy(x − 1)(x − 2)(x − 3)(x − 4) > 0, x ≠ 3} is"
7,"Let for the 9th term in the binomial expansion of (3 + 6x)ⁿ, in the increasing powers of 6x, to be the greatest for x = 3/2, the least value of n is n₀. If k is the ratio of the coefficient of x⁶ to the coefficient of x³, then k + n₀ is equal to:","Let for the 9th term in the binomial expansion of (3 + 6x)ⁿ, in the increasing powers of 6x, to be the greatest for x = 3/2, the least value of n is n₀."
8,"For some n ≠ 10, let the coefficients of the 5th, 6th and 7th terms in the binomial expansion of (1+x)^(n+4) be in A.P. Then the largest coefficient in the expansion of (1+x)^(n+4) is:","For some n ≠ 10, let the coefficients of the 5th, 6th and 7th terms in the binomial expansion of (1+x)^(n+4) be in A.P. Then the largest coefficient in the expansion of (1+x)^(n+4) is:"
9,"Let the equation of the circle, which touches x-axis at the point (a, 0), a > 0 and cuts off an intercept of length b on y-axis be x² + y² − αx + βy + γ = 0. If the circle lies below x-axis, then the ordered pair (2a, b²) is equal to","Let the equation of the circle, which touches x-axis at the point (a, 0), a > 0 and cuts off an intercept of length b on y-axis be x² + y² − αx + βy + γ = 0"


In [ ]:
print(f"Total count of True exemplars: {df_pyq_raw['is_exemplar'].sum()}")


Total count of True exemplars: 993




---



03. Source Data Profiling (df_pyq_raw)

In [ ]:
# Count the occurrences of each question type
type_counts = df_pyq_raw['question_type'].value_counts()

print("--- Total Question Type Distribution ---")
print(type_counts)

# To see it as a percentage (essential for EDA)
print("\n--- Percentage Distribution ---")
print(df_pyq_raw['question_type'].value_counts(normalize=True) * 100)

--- Total Question Type Distribution ---
question_type
MCQ          3367
Numerical    1114
Name: count, dtype: int64

--- Percentage Distribution ---
question_type
MCQ          75.139478
Numerical    24.860522
Name: proportion, dtype: float64


In [ ]:
# Calculate length of question text
df_pyq_raw['text_length'] = df_pyq_raw['question_text'].str.len()

# Check for outliers
print("--- Text Length Statistics ---")
print(df_pyq_raw['text_length'].describe())

# Functional Knowledge:
# If you have questions with length < 10, they are likely just images/symbols
# that failed extraction. If > 2000, they might be long "Match the Column" questions.

--- Text Length Statistics ---
count    4481.000000
mean      153.950234
std        64.822928
min        25.000000
25%       108.000000
50%       146.000000
75%       190.000000
max       526.000000
Name: text_length, dtype: float64


In [ ]:
# View ALL questions under 30 characters
df_short = df_pyq_raw[df_pyq_raw['text_length'] < 100].sort_values('text_length')

df_short.to_csv("df_short_math.csv",index=False)

print(f"Total questions under 30 characters: {len(df_short)}")
pd.set_option('display.max_rows', None)
pd.set_option('display.max_colwidth', None)
display(df_short[['question_id', 'question_text', 'text_length']].head(20))


Total questions under 30 characters: 923


,question_id,question_text,text_length
3142,Q_M_18_138,∑ₖ₌₀⁶ ⁵¹⁻ᵏC₃ is equal to:,25
4259,Q_M_25_031,The value of cot(π/24) is,25
783,Q_M_04_074,∑ₖ₌₀²⁰ (²⁰Cₖ)² is equal to:,27
1406,Q_M_07_157,∫₋π^π |π - |x|| dx is equal to:,31
1990,Q_M_10_079,The inverse of y = 5^(log x) is,31
769,Q_M_04_060,∑ᵣ₌₁²⁰ (r² + 1)(r!) is equal to:,32
1436,Q_M_07_187,The value of ∫₀^π |cos x|³ dx is:,33
772,Q_M_04_063,"∑ᵢ,ⱼ₌₀,ᵢ≠ⱼⁿ ⁿCᵢ · ⁿCⱼ is equal to:",34
4255,Q_M_25_027,The value of 2sin 12° − sin 72° is,34
768,Q_M_04_059,The value of ∑ᵣ₌₀²² ²²Cᵣ · ²³Cᵣ is:,35


In [ ]:
# 1. Filter for Mathematics only
math_questions = df_pyq_raw[df_pyq_raw['subject'] == 'Mathematics']

# 2. Group by topic and count the number of rows
topic_distribution = math_questions.groupby('topic')['question_id'].count().sort_values(ascending=False)

# 3. Print the result
print("--- Question Count per Topic (Mathematics) ---")
print(topic_distribution.to_string())

exemplar_distribution = math_questions[math_questions['is_exemplar'] == True].groupby('topic')['question_id'].count().sort_values(ascending=False)

print("\n--- 'Solvable' Exemplar Count per Topic (Mathematics) ---")

--- Question Count per Topic (Mathematics) ---
topic
3D Geometry                                  335
Matrices And Determinants                    324
Definite Integration                         278
Sequences And Series                         265
Limits Continuity And Differentiability      236
Binomial Theorem                             218
Application Of Derivatives                   218
Differential Equations                       214
Permutations And Combinations                199
Probability                                  199
Vector Algebra                               197
Complex Numbers                              172
Straight Lines And Pair Of Straight Lines    168
Quadratic Equation And Inequalities          164
Area Under The Curves                        157
Circle                                       150
Functions                                    145
Parabola                                     133
Statistics                                   122
Sets And Relatio



---



04. Options & Answer Key Quality Checks

In [ ]:
# @title
import os
import json
import glob
import pandas as pd

# 1. Define folder path and get all JSON files
folder_path = '/content/LECO/options_files_output'
file_list = glob.glob(os.path.join(folder_path, '*.json'))

all_options_data = []

# 2. Extract data and Topic from the file name
for file_path in file_list:
    # Extract filename (e.g., "Vector Algebra_batch_1_processed.json")
    file_name = os.path.basename(file_path)

    # Split by '_batch_' to get the topic name (e.g., "Vector Algebra")
    topic_name = file_name.split('_batch_')[0]

    with open(file_path, 'r', encoding='utf-8') as file:
        json_data = json.load(file)

        for question in json_data:
            row = {
                'question_id': question.get('question_id'),
                'topic': topic_name,  # Added topic right here from the filename
                'correct_answer': question.get('correct_answer'),
                'solution_steps': question.get('solution_steps', []),
                'wrong_options': question.get('wrong_options', []),
                'unsolvable_reason': question.get('unsolvable_reason', None)
            }
            all_options_data.append(row)

# 3. Create the options output dataframe
df_options_output = pd.DataFrame(all_options_data)

# 4. Group by the extracted topic to see the distribution in the PROCESSED files
topic_distribution_processed = df_options_output.groupby('topic')['question_id'].count().sort_values(ascending=False)

print("--- Question Count per Topic (Processed Options Output) ---")
print(topic_distribution_processed.to_string())
print(f"\nTotal questions processed across all topics: {len(df_options_output)}")

--- Question Count per Topic (Processed Options Output) ---
topic
3D Geometry                                  690
Permutations_And_Combinations                398
Matrices and Determinants                    324
Definite Integration                         278
Limits Continuity and Differentiability      236
Binomial Theorem                             218
Application Of Derivatives                   218
Sequences And Series                         215
Differential Equations                       214
Permutations and Combinations                199
Probability                                  199
Vector Algebra                               197
Inverse Trigonometric Functions              185
Complex Numbers                              172
Straight Lines And Pair Of Straight Lines    168
Quadratic Equation and Inequalities          164
Circle                                       150
Functions                                    145
Area under the curve                         137
Par

In [ ]:
# @title
import os
import pandas as pd

# ✨ THE FIX: Drop 'topic' from options_output, and take it from pyq_raw instead!
df_merged = pd.merge(
    df_options_output[['question_id', 'solution_steps']],         # <-- Removed 'topic' from here
    df_pyq_raw[['question_id', 'question_text', 'topic']],        # <-- Added 'topic' here!
    on='question_id',
    how='inner'
)

# Now, df_merged has the PERFECT topic names directly from the raw data.
# The rest of your batching code will now work flawlessly!

# Preserve topic order from df_pyq_raw
topic_order = df_pyq_raw[df_pyq_raw['subject'] == 'Mathematics']['topic'].unique()
df_merged['topic_order'] = df_merged['topic'].map({t: i for i, t in enumerate(topic_order)})
df_merged = df_merged.sort_values(['topic_order', 'question_id']).drop(columns='topic_order')

output_dir = "/content/merged_solution_batches_75"
os.makedirs(output_dir, exist_ok=True)
BATCH_SIZE = 75

print("--- Starting Batch Generation ---")
total_batches_created = 0

for topic in topic_order:
    topic_data = df_merged[df_merged['topic'] == topic]
    if topic_data.empty:
        continue
    total_q = len(topic_data)

    for i in range(0, total_q, BATCH_SIZE):
        batch_df = topic_data.iloc[i : i + BATCH_SIZE]
        batch_num = (i // BATCH_SIZE) + 1
        total_batches_created += 1
        safe_name = str(topic).replace(' ', '_').replace('/', '_')
        filename = f"{output_dir}/{safe_name}_batch_{batch_num}.json"
        batch_df[['question_id', 'topic', 'question_text', 'solution_steps']].to_json(
            filename, orient='records', indent=4, force_ascii=False
        )

    full_batches = total_q // BATCH_SIZE
    remainder = total_q % BATCH_SIZE
    if remainder == 0:
        dist_text = f"{full_batches} batches of {BATCH_SIZE}"
    elif full_batches == 0:
        dist_text = f"1 batch of {remainder}"
    else:
        dist_text = f"{full_batches} batches of {BATCH_SIZE}, 1 batch of {remainder}"
    print(f"[{str(topic):40}] Total Questions: {total_q:3} | {dist_text}")

print(f"\n✅ Done! {total_batches_created} total batches saved.")

--- Starting Batch Generation ---
[3D Geometry                             ] Total Questions: 690 | 9 batches of 75, 1 batch of 15
[Application Of Derivatives              ] Total Questions: 218 | 2 batches of 75, 1 batch of 68
[Area Under The Curves                   ] Total Questions: 157 | 2 batches of 75, 1 batch of 7
[Binomial Theorem                        ] Total Questions: 218 | 2 batches of 75, 1 batch of 68
[Circle                                  ] Total Questions: 150 | 2 batches of 75
[Complex Numbers                         ] Total Questions: 172 | 2 batches of 75, 1 batch of 22
[Definite Integration                    ] Total Questions: 278 | 3 batches of 75, 1 batch of 53
[Differential Equations                  ] Total Questions: 214 | 2 batches of 75, 1 batch of 64
[Differentiation                         ] Total Questions:  72 | 1 batch of 72
[Ellipse                                 ] Total Questions:  98 | 1 batches of 75, 1 batch of 23
[Functions                   

In [ ]:
duplicate_id_count = df_options_output.duplicated(subset=['question_id']).sum()
print(f"Number of duplicate question_ids: {duplicate_id_count}")

Number of duplicate question_ids: 888


In [ ]:
all_dups = df_options_output[df_options_output.duplicated(subset=['question_id'], keep=False)]

# Sort them by ID so the copies are right next to each other
all_dups_sorted = all_dups.sort_values(by='question_id')

# Display the first 10 rows (which will be 5 pairs of duplicates)
display(all_dups_sorted.head(10))

,question_id,topic,correct_answer,solution_steps,wrong_options,unsolvable_reason
1323,Q_M_01_001,3D Geometry,7,"[{'step': 1, 'action': 'Identify the given point P as (1, 2, a) and find the intersection of the given lines with the main line M.', 'concept_used': 'Point of intersection'}, {'step': 2, 'action': 'Find the intersection point A of L1 and M, and B of L2 and M, yielding A(4, 6, 4) and B(0, -2, 0).', 'concept_used': 'Parametric coordinates'}, {'step': 3, 'action': 'Equate the distance PA to PB as given, leading to an equation in a.', 'concept_used': 'Distance formula'}, {'step': 4, 'action': 'Solve for a to get a = 3.', 'concept_used': 'Algebraic simplification'}, {'step': 5, 'action': 'Determine parameters b and c using proportional direction ratios, resulting in b=1 and c=3, making a+b+c=7.', 'concept_used': 'Direction ratios'}]","[5, 8, 9]",None
3572,Q_M_01_001,3D Geometry,7,"[{'step': 1, 'action': 'Identify the given point P as (1, 2, a) and find the intersection of the given lines with the main line M.', 'concept_used': 'Point of intersection'}, {'step': 2, 'action': 'Find the intersection point A of L1 and M, and B of L2 and M, yielding A(4, 6, 4) and B(0, -2, 0).', 'concept_used': 'Parametric coordinates'}, {'step': 3, 'action': 'Equate the distance PA to PB as given, leading to an equation in a.', 'concept_used': 'Distance formula'}, {'step': 4, 'action': 'Solve for a to get a = 3.', 'concept_used': 'Algebraic simplification'}, {'step': 5, 'action': 'Determine parameters b and c using proportional direction ratios, resulting in b=1 and c=3, making a+b+c=7.', 'concept_used': 'Direction ratios'}]","[5, 8, 9]",None
1324,Q_M_01_002,3D Geometry,-2,"[{'step': 1, 'action': 'Extract points A(-1, 2, 4) and B(0, 1, 1) and direction vectors d1=(α, -1, -α) and d2=(α, 2, 2α) for the two lines.', 'concept_used': 'Line equation parsing'}, {'step': 2, 'action': 'Compute the cross product d1 x d2 = (0, -3α², 3α) and its magnitude.', 'concept_used': 'Cross product'}, {'step': 3, 'action': 'Calculate the scalar triple product of AB and (d1 x d2).', 'concept_used': 'Dot product'}, {'step': 4, 'action': 'Equate the shortest distance formula to 2 and square both sides.', 'concept_used': 'Shortest distance between skew lines'}, {'step': 5, 'action': 'Solve the resulting quadratic equation 3α² + 6α - 5 = 0 to find the sum of roots.', 'concept_used': 'Sum of roots of quadratic equation'}]","[2, 3, -3]",None
3573,Q_M_01_002,3D Geometry,-2,"[{'step': 1, 'action': 'Extract points A(-1, 2, 4) and B(0, 1, 1) and direction vectors d1=(α, -1, -α) and d2=(α, 2, 2α) for the two lines.', 'concept_used': 'Line equation parsing'}, {'step': 2, 'action': 'Compute the cross product d1 x d2 = (0, -3α², 3α) and its magnitude.', 'concept_used': 'Cross product'}, {'step': 3, 'action': 'Calculate the scalar triple product of AB and (d1 x d2).', 'concept_used': 'Dot product'}, {'step': 4, 'action': 'Equate the shortest distance formula to 2 and square both sides.', 'concept_used': 'Shortest distance between skew lines'}, {'step': 5, 'action': 'Solve the resulting quadratic equation 3α² + 6α - 5 = 0 to find the sum of roots.', 'concept_used': 'Sum of roots of quadratic equation'}]","[2, 3, -3]",None
1325,Q_M_01_003,3D Geometry,10 / (3√38),"[{'step': 1, 'action': 'Express n from the first equation as n = 4l + m.', 'concept_used': 'System of equations'}, {'step': 2, 'action': 'Substitute n into the second equation to get a homogeneous quadratic in l and m: 40l² + 21lm + 2m² = 0.', 'concept_used': 'Algebraic substitution'}, {'step': 3, 'action': 'Solve for l/m to obtain two sets of direction ratios: (-2, 5, -3) and (-1, 8, 4).', 'concept_used': 'Quadratic factorization'}, {'step': 4, 'action': 'Use the dot product to find the cosine of the angle between the two direction vectors.', 'concept_used': 'Angle between vectors'}]","[5 / (3√38), 10 / (9√38), 5 / (2√38)]",None
3574,Q_M_01_003,3D Geometry,10 / (3√38),"[{'step': 1, 'action': 'Express n from the first equa

In [ ]:
# 1. Isolate ONLY the extra duplicated rows (the 400 rows)
extra_copies_df = df_options_output[df_options_output.duplicated(subset=['question_id'], keep='first')]

# 2. Define your topic column name (change 'topic' if yours is named differently!)
topic_col = 'topic'

# 3. Print the distribution
if topic_col in extra_copies_df.columns:
    print(f"--- Topic Distribution for the {len(extra_copies_df)} Extra Duplicates ---\n")
    display(extra_copies_df[topic_col].value_counts())
else:
    print(f"Oops! The column '{topic_col}' was not found. Please change 'topic_col' to your actual column name (like 'chapter' or 'subject').")

--- Topic Distribution for the 888 Extra Duplicates ---



,count
topic,
3D Geometry,355
Permutations_And_Combinations,199
Permutations and Combinations,199
Inverse Trigonometric Functions,100
Indefinite Integrals,35


In [ ]:
# 1. Filter the extra copies for just the 'Probability' topic
prob_dups = extra_copies_df[extra_copies_df['topic'] == 'Permutations and Combinations']

# 2. Extract the question IDs and convert them to a list
prob_dup_ids = prob_dups['question_id'].tolist()

# 3. Print the results
print(f"Here are the {len(prob_dup_ids)} duplicated Question IDs for Probability:\n")
print(prob_dup_ids)

Here are the 199 duplicated Question IDs for Probability:

['Q_M_19_001', 'Q_M_19_002', 'Q_M_19_003', 'Q_M_19_004', 'Q_M_19_005', 'Q_M_19_006', 'Q_M_19_007', 'Q_M_19_008', 'Q_M_19_009', 'Q_M_19_010', 'Q_M_19_011', 'Q_M_19_012', 'Q_M_19_013', 'Q_M_19_014', 'Q_M_19_015', 'Q_M_19_016', 'Q_M_19_017', 'Q_M_19_018', 'Q_M_19_019', 'Q_M_19_020', 'Q_M_19_021', 'Q_M_19_022', 'Q_M_19_023', 'Q_M_19_024', 'Q_M_19_025', 'Q_M_19_026', 'Q_M_19_027', 'Q_M_19_028', 'Q_M_19_029', 'Q_M_19_030', 'Q_M_19_031', 'Q_M_19_032', 'Q_M_19_033', 'Q_M_19_034', 'Q_M_19_035', 'Q_M_19_036', 'Q_M_19_037', 'Q_M_19_038', 'Q_M_19_039', 'Q_M_19_040', 'Q_M_19_041', 'Q_M_19_042', 'Q_M_19_043', 'Q_M_19_044', 'Q_M_19_045', 'Q_M_19_046', 'Q_M_19_047', 'Q_M_19_048', 'Q_M_19_049', 'Q_M_19_050', 'Q_M_19_151', 'Q_M_19_152', 'Q_M_19_153', 'Q_M_19_154', 'Q_M_19_155', 'Q_M_19_156', 'Q_M_19_157', 'Q_M_19_158', 'Q_M_19_159', 'Q_M_19_160', 'Q_M_19_161', 'Q_M_19_162', 'Q_M_19_163', 'Q_M_19_164', 'Q_M_19_165', 'Q_M_19_166', 'Q_M_19_167', 'Q



---



05. Options Dataset Cleaning & Standardization

In [ ]:
# 1. Create a separate dataframe for rows where 'correct_answer' is null
df_unsolvable = df_options_output[df_options_output['correct_answer'].isna()].copy()

# 2. Get the total count
unsolvable_count = len(df_unsolvable)

# 3. Print the total count
print(f"Total questions with 'null' correct answer: {unsolvable_count}\n")

# 4. Optional but helpful: Show the distribution of these unsolvable questions per topic
if unsolvable_count > 0:
    print("--- Unsolvable Questions Count per Topic ---")
    print(df_unsolvable['topic'].value_counts().to_string())


else:
    print("Great news! There are 0 questions with a null correct_answer.")

Total questions with 'null' correct answer: 118

--- Unsolvable Questions Count per Topic ---
topic
Limits Continuity and Differentiability      26
Differential Equations                       22
Definite Integration                         12
Complex Numbers                               6
Vector Algebra                                5
Straight Lines And Pair Of Straight Lines     5
3D Geometry                                   4
Differentiation                               4
Probability                                   4
Application Of Derivatives                    4
Permutations and Combinations                 4
Indefinite Integrals                          4
Ellipse                                       3
Sequences And Series                          3
Functions                                     3
Permutations_And_Combinations                 2
Matrices and Determinants                     2
Trigonometric Ratio And Identites             1
Sets And Relations                  

In [ ]:

df_unsolvable_with_text = pd.merge(
    df_unsolvable[['question_id', 'topic', 'unsolvable_reason']],
    df_pyq_raw[['question_id', 'question_text']],
    on='question_id',
    how='left'
)

# 2. Display the first few rows to verify
df_unsolvable_with_text
df_unsolvable_with_text.to_csv('unsolvable_questions_with_text.csv', index=False)

In [ ]:
import pandas as pd
import json

# 1. Open and load the specific unsolved.json file
# (Make sure to upload 'unsolved.json' to your Colab files first!)
with open('/content/LECO/unsolved.json', 'r', encoding='utf-8') as f:
    unsolved_data = json.load(f)

# 2. Create the new DataFrame
df_unsolved_reanalyzed = pd.DataFrame(unsolved_data)


# 3. Separate the resolved and still-null questions
df_resolved = df_unsolved_reanalyzed[df_unsolved_reanalyzed['correct_answer'].notna()]
df_still_null = df_unsolved_reanalyzed[df_unsolved_reanalyzed['correct_answer'].isna()]

# 4. Print the counts
print(f"Total questions in this file: {len(df_unsolved_reanalyzed)}")
print(f"Total RESOLVED: {len(df_resolved)}")
print(f"Total STILL NULL: {len(df_still_null)}\n")

# # 5. Display ALL resolved questions (forcing Pandas not to truncate)
# print("--- ALL RESOLVED QUESTIONS ---")
# with pd.option_context('display.max_rows', None):  # 'None' means show every single row
#     # Just showing a few key columns so it's easy to read on screen
#     display(df_resolved[['question_id', 'correct_answer', 'assumed_correction']])



Total questions in this file: 113
Total RESOLVED: 99
Total STILL NULL: 14



In [ ]:
import pandas as pd

# 1. First, make sure we only have the successfully solved ones
df_newly_solved = df_unsolved_reanalyzed[df_unsolved_reanalyzed['correct_answer'].notna()].copy()

# 2. Add 'assumed_correction' column to the main df if it doesn't exist yet
if 'assumed_correction' not in df_options_output.columns:
    df_options_output['assumed_correction'] = None

# ---- FIX IS HERE ----
# Drop any duplicate question_ids so Pandas has a clean 1-to-1 mapping
df_options_output = df_options_output.drop_duplicates(subset=['question_id'], keep='first')
df_newly_solved = df_newly_solved.drop_duplicates(subset=['question_id'], keep='first')
# ---------------------

# 3. Temporarily set 'question_id' as the index for both DataFrames
df_options_output.set_index('question_id', inplace=True)
df_newly_solved.set_index('question_id', inplace=True)

# 4. Specify the exact columns we want to replace
cols_to_replace = ['correct_answer', 'solution_steps', 'wrong_options', 'assumed_correction']

# 5. Overwrite those columns in df_options_output with the data from df_newly_solved
df_options_output.update(df_newly_solved[cols_to_replace])

# 6. Put 'None' in the place of unsolvable_reason for all these newly solved questions
for q_id in df_newly_solved.index:
    if q_id in df_options_output.index:
        df_options_output.at[q_id, 'unsolvable_reason'] = None

# 7. Reset the indexes so 'question_id' goes back to being a normal column
df_options_output.reset_index(inplace=True)
df_newly_solved.reset_index(inplace=True)

print(f"✅ Successfully updated {len(df_newly_solved)} questions in df_options_output!")

# --- Optional: Let's prove it worked by checking that specific question! ---
print("\n--- Checking Q_M_26_125 ---")
display(df_options_output[df_options_output['question_id'] == 'Q_M_26_125'])

✅ Successfully updated 98 questions in df_options_output!

--- Checking Q_M_26_125 ---


,question_id,topic,correct_answer,solution_steps,wrong_options,unsolvable_reason,assumed_correction
1207,Q_M_26_125,Vector Algebra,5π/6,"[{'step': 1, 'action': 'Use BAC-CAB rule: a⃗ × (b⃗ × c⃗) = (a⃗·c⃗)b⃗ − (a⃗·b⃗)c⃗ = (√3/2)(b⃗ + c⃗).', 'concept_used': 'Vector triple product identity'}, {'step': 2, 'action': 'Since b⃗ not parallel to c⃗, equate coefficients: a⃗·c⃗ = √3/2 and −a⃗·b⃗ = √3/2, so a⃗·b⃗ = −√3/2.', 'concept_used': 'Linear independence of vectors'}, {'step': 3, 'action': 'Since all are unit vectors: |a⃗||b⃗|cosθ = a⃗·b⃗ = −√3/2, so cosθ = −√3/2.', 'concept_used': 'Dot product formula for unit vectors'}, {'step': 4, 'action': 'θ = arccos(−√3/2) = 5π/6 (150°).', 'concept_used': 'Inverse cosine evaluation'}]","[π/6, 2π/3, π/3]",None,Typo confirmed: (3/2) should be (√3/2). Corrected: a⃗ × (b⃗ × c⃗) = (√3/2)(b⃗ + c⃗).


In [ ]:
# Count how many rows in the main dataframe STILL have a null correct_answer
remaining_nulls = df_options_output['correct_answer'].isna().sum()

print(f"Total questions that STILL have a 'null' correct answer: {remaining_nulls}")
print(f"Total fully solved questions: {len(df_options_output) - remaining_nulls}")

Total questions that STILL have a 'null' correct answer: 16
Total fully solved questions: 4415


In [ ]:
# 1. Keep ONLY the rows where 'correct_answer' is NOT null
# (This permanently removes the broken questions)
df_options_output = df_options_output[df_options_output['correct_answer'].notna()].copy()

# 2. Drop the 'unsolvable_reason' and 'assumed_correction' columns completely
columns_to_drop = ['unsolvable_reason', 'assumed_correction']
df_options_output.drop(columns=columns_to_drop, inplace=True, errors='ignore')

# 3. Print a final confirmation
print(f"✅ Cleanup complete!")
print(f"Final number of valid questions: {len(df_options_output)}")
print(f"Current columns in df_options_output: {df_options_output.columns.tolist()}")

# Optional: View the clean data
display(df_options_output.head(2))

✅ Cleanup complete!
Final number of valid questions: 4415
Current columns in df_options_output: ['question_id', 'topic', 'correct_answer', 'solution_steps', 'wrong_options']


,question_id,topic,correct_answer,solution_steps,wrong_options
0,Q_M_02_021,Application Of Derivatives,"(1/e, ∞)","[{'step': 1, 'action': 'Take the natural logarithm of f(x) = x^x to get ln y = x ln x.', 'concept_used': 'Logarithmic differentiation'}, {'step': 2, 'action': 'Differentiate with respect to x to find f'(x) = x^x (1 + ln x).', 'concept_used': 'Product rule'}, {'step': 3, 'action': 'Set f'(x) > 0 for strictly increasing, which requires 1 + ln x > 0 since x^x > 0.', 'concept_used': 'Increasing functions'}, {'step': 4, 'action': 'Solve the inequality to get ln x > -1, meaning x > 1/e.', 'concept_used': 'Logarithmic inequalities'}]","[(0, 1/e), (1, ∞), (-∞, 1/e)]"
1,Q_M_02_022,Application Of Derivatives,72,"[{'step': 1, 'action': 'Let the angle between AB and PQ be θ. Then the sides of PQRS are a = 4cosθ + 2sinθ and b = 4sinθ + 2cosθ.', 'concept_used': 'Trigonometric projection'}, {'step': 2, 'action': 'Express the area of PQRS as a*b = 10sin(2θ) + 8.', 'concept_used': 'Area of rectangle'}, {'step': 3, 'action': 'Find the maximum area by setting sin(2θ) = 1, which gives θ = π/4.', 'concept_used': 'Maximization of trigonometric functions'}, {'step': 4, 'action': 'Substitute θ = π/4 to get a = 3√2 and b = 3√2.', 'concept_used': 'Trigonometric values'}, {'step': 5, 'action': 'Calculate (a + b)² = (6√2)² = 72.', 'concept_used': 'Algebraic simplification'}]","[64, 100, 80]"


In [ ]:
missing_topics = [
    'Area Under The Curves',
    'Limits Continuity And Differentiability',
    'Matrices And Determinants',
    'Permutations And Combinations',
    'Quadratic Equation And Inequalities'
]

print("--- Checking if the missing topics exist in df_options_output ---")
for t in missing_topics:
    # We check using string contains to avoid case-sensitivity issues
    count = df_options_output['topic'].str.contains(t, case=False, na=False).sum()
    print(f"{t}: {count} questions found")

print("\n--- Total rows in both datasets ---")
print(f"Total rows in df_options_output: {len(df_options_output)}")
print(f"Total rows in df_pyq_raw: {len(df_pyq_raw)}")

--- Checking if the missing topics exist in df_options_output ---
Area Under The Curves: 0 questions found
Limits Continuity And Differentiability: 234 questions found
Matrices And Determinants: 324 questions found
Permutations And Combinations: 0 questions found
Quadratic Equation And Inequalities: 164 questions found

--- Total rows in both datasets ---
Total rows in df_options_output: 4415
Total rows in df_pyq_raw: 4481




---



06. Import Knowledge Graph Annotations

In [ ]:
import pandas as pd
import glob

# 1. Get all CSV files from the nodes folder
node_files = glob.glob('/content/LECO/nodes_files output/*.csv')

# 2. Read them all and combine them into one DataFrame
df_nodes = pd.concat((pd.read_csv(f, on_bad_lines='skip') for f in node_files), ignore_index=True)

# 3. Check the results
print(f"Total questions loaded: {len(df_nodes)}")
display(df_nodes.head())

Total questions loaded: 4481


,question_id,topic,node_names
0,Q_M_15_001,Logarithm,Log-equation-to-polynomial reduction
1,Q_M_15_002,Logarithm,Log-inequality with variable base;Domain-constrained solution counting
2,Q_M_15_003,Logarithm,Base-argument entanglement resolution
3,Q_M_15_004,Logarithm,Logarithmic pre-processing feeding a downstream problem
4,Q_M_15_005,Logarithm,Log-equation-to-polynomial reduction;Domain-constrained solution counting


In [ ]:
import csv
import pandas as pd
import os

def safe_read_concept_csv(filepath):
    """
    Reads a concept CSV where concept_names may contain unquoted commas.
    Correctly re-joins any extra fields back into concept_names.
    """
    rows = []
    with open(filepath, 'r', encoding='utf-8') as f:
        reader = csv.reader(f)
        header = next(reader)  # skip header row
        for raw_row in reader:
            if len(raw_row) < 3:
                continue
            q_id    = raw_row[0].strip()
            topic   = raw_row[1].strip()
            # Everything from column 2 onward is the concept_names — rejoin with comma
            concept = ','.join(raw_row[2:]).strip()
            rows.append({
                'question_id':   q_id,
                'topic':         topic,
                'concept_names': concept,
                'file_name':     os.path.basename(filepath)
            })
    return pd.DataFrame(rows)

# ─── Use this instead of pd.read_csv for ALL concept files ───────────────────
CONCEPTS_FOLDER = "/content/LECO/concepts_files_output"  # your actual path

all_dfs = []
for fname in sorted(os.listdir(CONCEPTS_FOLDER)):
    if fname.endswith('.csv'):
        df_temp = safe_read_concept_csv(os.path.join(CONCEPTS_FOLDER, fname))
        all_dfs.append(df_temp)

df_concepts = pd.concat(all_dfs, ignore_index=True)

# Verify
print(f"Total rows loaded     : {len(df_concepts)}")
print(f"Files loaded          : {df_concepts['file_name'].nunique()}")

# Check zero weird topics remain
valid_topics = set(df_pyq_raw[df_pyq_raw['subject'] == 'Mathematics']['topic'].unique())
weird = df_concepts[~df_concepts['topic'].isin(valid_topics)]
print(f"Weird topic rows left : {len(weird)}")  # should be 0

Total rows loaded     : 4481
Files loaded          : 57
Weird topic rows left : 235


In [ ]:
import pandas as pd
import glob

# 1. Get all CSV files from the concepts folder
error_files = glob.glob('/content/LECO/error_files_output/*.csv')

# 2. Read them all and combine them into one DataFrame
df_errors = pd.concat((pd.read_csv(f, on_bad_lines='skip') for f in error_files), ignore_index=True)

# 3. Check the results
print(f"Total questions loaded: {len(df_errors)}")
display(df_errors.head())

Total questions loaded: 4465


,question_id,topic,error_names,error_types
0,Q_M_15_001,Logarithm,Product-of-roots vs. sum-of-roots confusion,NaN
1,Q_M_15_002,Logarithm,Squared-argument inequality flip neglect;Domain-blind solving,NaN
2,Q_M_15_003,Logarithm,Reciprocal-log substitution blindness;Domain-blind solving,NaN
3,Q_M_15_004,Logarithm,Domain-blind solving,NaN
4,Q_M_15_005,Logarithm,Reciprocal-log substitution blindness;Domain-blind solving,NaN




---



07. Annotation Integrity Checks

In [ ]:
duplicate_id_count = df_nodes.duplicated(subset=['question_id']).sum()
print(f"Number of duplicate question_ids: {duplicate_id_count}")

Number of duplicate question_ids: 0


In [ ]:
duplicate_id_count = df_concepts.duplicated(subset=['question_id']).sum()
print(f"Number of duplicate question_ids: {duplicate_id_count}")

Number of duplicate question_ids: 0


In [ ]:
df_concepts.shape

(4481, 4)

In [ ]:
duplicate_id_count = df_errors.duplicated(subset=['question_id']).sum()
print(f"Number of duplicate question_ids: {duplicate_id_count}")

Number of duplicate question_ids: 0


In [ ]:
df_errors.drop(columns=['error_types'], inplace=True)

In [ ]:

valid_topics = set(df_pyq_raw[df_pyq_raw['subject'] == 'Mathematics']['topic'].unique())

# ─── Split df_concepts into clean vs weird ────────────────────────────────────
df_normal  = df_concepts[df_concepts['topic'].isin(valid_topics)].copy()
df_weird   = df_concepts[~df_concepts['topic'].isin(valid_topics)].copy()

topic_lookup = {t.lower(): t for t in valid_topics}

mask_A = df_concepts['topic'].str.lower().map(topic_lookup).isin(valid_topics) & \
         ~df_concepts['topic'].isin(valid_topics)

print(f"Rows to fix (Pattern A): {mask_A.sum()}")
df_concepts.loc[mask_A, 'topic'] = df_concepts.loc[mask_A, 'topic'].str.strip().str.lower().map(topic_lookup)

# Verify
still_wrong = df_concepts.loc[mask_A & ~df_concepts['topic'].isin(valid_topics)]
print(f"Still wrong after fix: {len(still_wrong)}")  # should be 0
print("Pattern A done ✓")

Rows to fix (Pattern A): 235
Still wrong after fix: 0
Pattern A done ✓


In [ ]:
import pandas as pd

# # ─────────────────────────────────────────────
# # STAGE 1 — Verify concat row counts
# # ─────────────────────────────────────────────
# print("=== RAW SHAPES ===")
# print(f"df_concepts : {df_concepts.shape}")
# print(f"df_nodes    : {df_nodes.shape}")
# print(f"df_pyq_raw  : {df_pyq_raw.shape}")

# # How many unique files actually loaded?
# print("\n=== FILES LOADED ===")
# print(f"Concept files : {df_concepts['file_name'].nunique()} / 57")
# print(f"Node files    : {df_nodes['file_name'].nunique()} / 57")

# # Any files loaded 0 rows?
# missing_concept_files = set(range(1, 58)) - set(...)  # adapt to your naming scheme
# files_with_zero = (
#     df_nodes.groupby("file_name").size()
#     .pipe(lambda s: s[s == 0])
# )
# print(f"Node files with 0 rows: {len(files_with_zero)}")

# # ─────────────────────────────────────────────
# # STAGE 2 — Deduplicate df_concepts
# # ─────────────────────────────────────────────
# n_before = len(df_concepts)
# df_concepts_clean = df_concepts.drop_duplicates(subset=["question_id", "topic"])
# n_after = len(df_concepts_clean)
# print(f"\n=== CONCEPT DEDUP ===")
# print(f"Rows before : {n_before}")
# print(f"Rows after  : {n_after}")
# print(f"Duplicates dropped : {n_before - n_after}")  # expect ~279

# ─────────────────────────────────────────────
# STAGE 3 — Diagnose missing rows in df_nodes
# # ─────────────────────────────────────────────
master_ids = set(df_pyq_raw["question_id"])
node_ids   = set(df_nodes["question_id"])
concept_ids = set(df_concepts['question_id'])

missing_in_nodes = master_ids - node_ids
extra_in_nodes   = node_ids - master_ids
missing_in_concepts = master_ids - concept_ids

print(f"\n=== NODE DIAGNOSIS ===")
print(f"question_ids in df_pyq_raw but NOT in df_nodes : {len(missing_in_nodes)}")
print(f"question_ids in df_pyq_raw but NOT in df_concept : {len(missing_in_concepts)}")


# Which topics/files are the missing IDs from?
if missing_in_nodes:
    missing_detail = df_pyq_raw[df_pyq_raw["question_id"].isin(missing_in_nodes)]
    print("\nMissing IDs breakdown by topic:")
    print(missing_detail["topic"].value_counts())

if missing_in_concepts:
    missing_detail = df_pyq_raw[df_pyq_raw["question_id"].isin(missing_in_concepts)]
    print("\nMissing IDs breakdown by topic:")
    print(missing_detail["topic"].value_counts())


# # STAGE 4 — Full reconciliation report
# # ─────────────────────────────────────────────
# concept_ids = set(df_concepts_clean["question_id"])

# missing_in_concepts = master_ids - concept_ids
# extra_in_concepts   = concept_ids - master_ids

# print(f"\n=== RECONCILIATION REPORT ===")
# print(f"Master IDs (df_pyq_raw)            : {len(master_ids)}")
# print(f"Missing from df_concepts_clean     : {len(missing_in_concepts)}")
# print(f"Missing from df_nodes              : {len(missing_in_nodes)}")
# print(f"Orphan IDs in df_concepts (not in master) : {len(extra_in_concepts)}")
# print(f"Orphan IDs in df_nodes    (not in master) : {len(extra_in_nodes)}")

# # Save reports for investigation
# report_missing_concepts = df_pyq_raw[df_pyq_raw["question_id"].isin(missing_in_concepts)][["question_id","topic","subject"]]
# report_missing_nodes    = df_pyq_raw[df_pyq_raw["question_id"].isin(missing_in_nodes)][["question_id","topic","subject"]]

# print("\nMissing from concepts — by topic:")
# print(report_missing_concepts["topic"].value_counts())

# print("\nMissing from nodes — by topic:")
# print(report_missing_nodes["topic"].value_counts())

# # ─────────────────────────────────────────────
# # OUTPUTS
# # ─────────────────────────────────────────────
# # df_concepts_clean  → deduplicated, reconciled
# # df_nodes           → unchanged pending Stage 3 fix
# # report_missing_*   → tells you exactly which topics/files to re-run


=== NODE DIAGNOSIS ===
question_ids in df_pyq_raw but NOT in df_nodes : 0
question_ids in df_pyq_raw but NOT in df_concept : 0


In [ ]:
# ─── Understand the 124 duplicates in df_concepts ────────────────────────────

# Check 1: Same question_id + same concept (TRUE duplicate — should not exist)
true_dupes = df_errors.duplicated(subset=['question_id', 'error_names'], keep=False)
print(f"True duplicates (same q_id + same errors): {true_dupes.sum()}")

# Check 2: Same question_id but different errorss (LEGITIMATE 1-to-N rows)
qid_counts = df_errors.groupby('question_id').size()
multi_errors_qids = qid_counts[qid_counts > 1]
print(f"question_ids with multiple errorss (expected): {len(multi_errors_qids)}")
print(f"Max errorss for a single question: {qid_counts.max()}")
print(f"Distribution:")
print(qid_counts.value_counts().sort_index().to_string())

# Check 3: Unique question_ids actually present
print(f"\nUnique question_ids in df_errorss : {df_errors['question_id'].nunique()}")
print(f"Total in df_pyq_raw                : 4481")
print(f"Missing from df_errorss           : {4481 - df_errors['question_id'].nunique()}")

True duplicates (same q_id + same errors): 0
question_ids with multiple errorss (expected): 0
Max errorss for a single question: 1
Distribution:
1    4465

Unique question_ids in df_errorss : 4465
Total in df_pyq_raw                : 4481
Missing from df_errorss           : 16


In [ ]:
# Get all unique question_ids
all_qids = set(df_pyq_raw['question_id'].unique())
errors_qids = set(df_errors['question_id'].unique())

# Find missing question_ids
missing_qids = all_qids - errors_qids

print(f"Total missing: {len(missing_qids)}")

# See them
print(missing_qids)

Total missing: 16
{'Q_M_19_192', 'Q_M_22_056', 'Q_M_19_188', 'Q_M_07_015', 'Q_M_14_216', 'Q_M_01_073', 'Q_M_07_012', 'Q_M_14_209', 'Q_M_07_041', 'Q_M_19_056', 'Q_M_18_099', 'Q_M_12_067', 'Q_M_07_007', 'Q_M_19_194', 'Q_M_07_011', 'Q_M_12_068'}


In [ ]:
# ─── df_errors full diagnostic ────────────────────────────────────────────────

# Step 1: Drop the 76 true duplicates immediately
df_errors_clean = df_errors.drop_duplicates(subset=['question_id'])
# adjust subset to whatever the key column(s) are in df_errors
# if it has multiple rows per question legitimately, use subset=['question_id', '<content_col>']

print(f"df_errors before dedup : {len(df_errors)}")
print(f"df_errors after dedup  : {len(df_errors_clean)}")
print(f"Dropped                : {len(df_errors) - len(df_errors_clean)}")

# Step 2: Identify the 247 missing q_ids
master_ids   = set(df_pyq_raw['question_id'])
error_ids    = set(df_errors_clean['question_id'])
missing_ids  = master_ids - error_ids
orphan_ids   = error_ids - master_ids

print(f"\nMissing from df_errors  : {len(missing_ids)}")
print(f"Orphan IDs (not in master): {len(orphan_ids)}")

# Step 3: Decode topic codes from question_id format Q_M_XX_YYY
# XX is the topic code — map it to topic name via df_pyq_raw
missing_df = df_pyq_raw[df_pyq_raw['question_id'].isin(missing_ids)][['question_id','topic']]

print(f"\nMissing IDs breakdown by topic:")
print(missing_df['topic'].value_counts().to_string())

# Step 4: Figure out which topic codes map to which topics
# (to confirm 01=Matrices, 21=?, 22=? etc.)
topic_code_map = (
    df_pyq_raw['question_id']
    .str.extract(r'Q_M_(\d+)_')
    .rename(columns={0: 'code'})
    .assign(topic=df_pyq_raw['topic'])
    .drop_duplicates()
    .sort_values('code')
)
print("\nTopic code → topic name mapping:")
print(topic_code_map.to_string())

# Step 5: Check if missing IDs cluster in specific batch ranges
# (to know if whole files are missing or just gaps)
missing_df['batch_num'] = missing_df['question_id'].str.extract(r'Q_M_\d+_(\d+)').astype(int)
print("\nMissing IDs — number range per topic:")
for topic, grp in missing_df.groupby('topic'):
    nums = grp['batch_num'].sort_values()
    print(f"  {topic}: {len(nums)} missing, range {nums.min()}–{nums.max()}, "
          f"consecutive: {list(nums)}")

df_errors before dedup : 4465
df_errors after dedup  : 4465
Dropped                : 0

Missing from df_errors  : 16
Orphan IDs (not in master): 0

Missing IDs breakdown by topic:
topic
Definite Integration                       5
Probability                                4
Indefinite Integrals                       2
Limits Continuity And Differentiability    2
3D Geometry                                1
Permutations And Combinations              1
Sets And Relations                         1

Topic code → topic name mapping:
     code                                      topic
1528   00                     Differential Equations
0      01                                3D Geometry
335    02                 Application Of Derivatives
553    03                      Area Under The Curves
710    04                           Binomial Theorem
928    05                                     Circle
1078   06                            Complex Numbers
1250   07                       Definite 

In [ ]:
# ─── Final df_errors clean state ──────────────────────────────────────────────

# Confirm zero true dupes (already verified above — dedup dropped nothing)
# So df_errors is already clean, just has 16 fewer questions than master

# Document the 16 dropped question_ids for audit trail
missing_error_ids = master_ids - set(df_errors['question_id'])
df_errors_missing_log = df_pyq_raw[
    df_pyq_raw['question_id'].isin(missing_error_ids)
][['question_id', 'topic', 'question_text']].reset_index(drop=True)

print(f"Questions with no error data (will be null in master): {len(df_errors_missing_log)}")
print(df_errors_missing_log[['question_id', 'topic']].to_string())

# Save the log so you have an audit trail
df_errors_missing_log.to_csv("errors_missing_16_questions.csv", index=False)
print("\ndf_errors is clean. Ready for master merge.")
print(f"Final shape: {df_errors.shape}")

Questions with no error data (will be null in master): 16
   question_id                                    topic
0   Q_M_01_073                              3D Geometry
1   Q_M_07_007                     Definite Integration
2   Q_M_07_011                     Definite Integration
3   Q_M_07_012                     Definite Integration
4   Q_M_07_015                     Definite Integration
5   Q_M_07_041                     Definite Integration
6   Q_M_12_067                     Indefinite Integrals
7   Q_M_12_068                     Indefinite Integrals
8   Q_M_14_209  Limits Continuity And Differentiability
9   Q_M_14_216  Limits Continuity And Differentiability
10  Q_M_18_099            Permutations And Combinations
11  Q_M_19_056                              Probability
12  Q_M_19_188                              Probability
13  Q_M_19_192                              Probability
14  Q_M_19_194                              Probability
15  Q_M_22_056                       Sets And 



---



Master Dataset Construction

In [ ]:
# @title
# Create a dictionary of the specific DataFrames mentioned in your snippet
dfs_to_check = {
    "df_concepts": df_concepts,
    "df_nodes": df_nodes,
    "df_errors": df_errors,
    "df_pyq_raw": df_pyq_raw,
    "df_options_output": df_options_output,
    # "df_master": df_master
}

# Loop through and print the columns for each one
for name, df in dfs_to_check.items():
    try:
        print(f"=== {name} ({len(df.columns)} columns) ===")
        print(df.columns.tolist())
        print("\n")
    except NameError:
        print(f"=== {name} ===")
        print("[DataFrame not found in memory]\n")

=== df_concepts (4 columns) ===
['question_id', 'topic', 'concept_names', 'file_name']


=== df_nodes (3 columns) ===
['question_id', 'topic', 'node_names']


=== df_errors (3 columns) ===
['question_id', 'topic', 'error_names']


=== df_pyq_raw (11 columns) ===
['question_id', 'topic', 'subject', 'year', 'shift', 'question_type', 'question_text', 'is_exemplar', 'exemplar_id', 'node_id', 'text_length']


=== df_options_output (5 columns) ===
['question_id', 'topic', 'correct_answer', 'solution_steps', 'wrong_options']




In [ ]:
# @title
# Create a dictionary of your dataframes
dataframes = {
    # "df_master": df_master,
    "df_dna_nodes": df_dna_nodes,
    "df_concepts": df_concepts,
    "df_error_tax": df_error_tax,
    "df_exemplars": df_exemplars,    # Added just in case you need it
    "df_dimensions": df_dimensions   # Added just in case you need it
}

# Loop through and print the columns for each one
for name, df in dataframes.items():
    print(f"=== {name} ({len(df.columns)} columns) ===")

    # Check if dataframe is empty (like df_concepts might be based on your logic)
    if df.empty and len(df.columns) == 0:
         print("[Empty DataFrame]")
    else:
        # Print columns as a clean list
        print(df.columns.tolist())
    print("\n")

=== df_dna_nodes (7 columns) ===
['node_id', 'node_name', 'cognitive_operation', 'subject', 'source_topic', 'reasoning', 'cognitive_demand']


=== df_concepts (4 columns) ===
['question_id', 'topic', 'concept_names', 'file_name']


=== df_error_tax (6 columns) ===
['error_id', 'topic', 'subject', 'error_name', 'description', 'error_type']


=== df_exemplars (6 columns) ===
['exemplar_id', 'node_id', 'excerpt_text', 'year', 'source_topic', 'match_text']


=== df_dimensions (7 columns) ===
['dimension_id', 'topic', 'subject', 'dimension_name', 'low_end', 'high_end', 'diagnostic_value']




In [ ]:
import pandas as pd
import re

# ==========================================
# DIAGNOSTIC CHECK
# ==========================================
print("=" * 55)
print("PRE-BUILD DIAGNOSTIC")
print("=" * 55)

# 1. Row counts
print("\n── Row counts ──────────────────────────────────────")
print(f"  df_pyq_raw       : {len(df_pyq_raw):>6} rows")
print(f"  df_options_output: {len(df_options_output):>6} rows")
print(f"  df_concepts      : {len(df_concepts):>6} rows")
print(f"  df_nodes         : {len(df_nodes):>6} rows")
print(f"  df_errors        : {len(df_errors):>6} rows")

# 2. question_id coverage
all_qids = set(df_pyq_raw['question_id'])
print("\n── question_id coverage ────────────────────────────")
for name, df in [('df_options_output', df_options_output),
                 ('df_concepts',       df_concepts),
                 ('df_nodes',          df_nodes),
                 ('df_errors',         df_errors)]:
    covered = set(df['question_id'])
    missing = all_qids - covered
    extra   = covered - all_qids
    print(f"  {name:<20} missing: {len(missing):>4}   extra: {len(extra):>4}")

# 3. Duplicates
print("\n── Duplicate question_ids ──────────────────────────")
for name, df in [('df_pyq_raw',        df_pyq_raw),
                 ('df_options_output',  df_options_output),
                 ('df_concepts',        df_concepts),
                 ('df_nodes',           df_nodes),
                 ('df_errors',          df_errors)]:
    dupes = df.duplicated('question_id').sum()
    print(f"  {name:<20} duplicates: {dupes:>4}")

# 4. Nulls in key join columns
print("\n── Nulls in key columns ────────────────────────────")
checks = [
    ('df_pyq_raw',        df_pyq_raw,        'question_id'),
    ('df_options_output', df_options_output, 'question_id'),
    ('df_concepts',       df_concepts,       'concept_names'),
    ('df_nodes',          df_nodes,          'node_names'),
    ('df_errors',         df_errors,         'error_names'),
    ('df_dna_nodes',      df_dna_nodes,      'node_id'),
    ('df_concepts_q',     df_concepts_q,     'concept_id'),
    ('df_error_tax',      df_error_tax,      'error_id'),
]
for name, df, col in checks:
    nulls = df[col].isna().sum()
    print(f"  {name:<20} {col:<15} nulls: {nulls:>4}")

# 5. Topic consistency
print("\n── Topic name variants (capitalisation check) ──────")
all_topics = pd.concat([
    df_pyq_raw['topic'],
    df_concepts['topic'],
    df_nodes['topic'],
    df_errors['topic'],
]).unique()
import collections
topic_groups = collections.defaultdict(list)
for t in all_topics:
    topic_groups[t.lower().strip()].append(t)
mismatches = {k: v for k, v in topic_groups.items() if len(v) > 1}
if mismatches:
    print("  WARNING — same topic, different capitalisation:")
    for k, v in mismatches.items():
        print(f"    {v}")
else:
    print("  ✅ All topic names consistent")

print("\n✅ Diagnostic complete — proceed if no critical warnings above")
print("=" * 55)

PRE-BUILD DIAGNOSTIC

── Row counts ──────────────────────────────────────
  df_pyq_raw       :   4481 rows
  df_options_output:   4415 rows
  df_concepts      :   4481 rows
  df_nodes         :   4481 rows
  df_errors        :   4465 rows

── question_id coverage ────────────────────────────
  df_options_output    missing:   66   extra:    0
  df_concepts          missing:    0   extra:    0
  df_nodes             missing:    0   extra:    0
  df_errors            missing:   16   extra:    0

── Duplicate question_ids ──────────────────────────
  df_pyq_raw           duplicates:    0
  df_options_output    duplicates:    0
  df_concepts          duplicates:    0
  df_nodes             duplicates:    0
  df_errors            duplicates:    0

── Nulls in key columns ────────────────────────────
  df_pyq_raw           question_id     nulls:    0
  df_options_output    question_id     nulls:    0
  df_concepts          concept_names   nulls:    0
  df_nodes             node_names      nu

In [ ]:


# ==========================================
# CELL 2: df_master
# ==========================================
def split_col(val):
    if pd.isna(val) or str(val).strip() in ['', '[]', 'nan']:
        return []
    return [x.strip() for x in str(val).split(';') if x.strip()]

df_options_merge = df_options_output.drop(columns=['topic'], errors='ignore')

df_master = (
    df_pyq_raw
    .drop(columns=['node_id'], errors='ignore')
    .merge(df_options_merge, on='question_id', how='left')
)

df_master = df_master.copy()

print(f"\n✅ df_master : {df_master.shape}")
print(f"   Columns: {df_master.columns.tolist()}")


# ==========================================
# CELL 3: BRIDGE TABLE 1 — df_question_nodes
# ==========================================
df_question_nodes = df_nodes.copy()
df_question_nodes['topic'] = df_question_nodes['topic'].str.replace('3D geometry', '3D Geometry', regex=False)
df_question_nodes['node_name'] = df_question_nodes['node_names'].apply(split_col)
df_question_nodes = df_question_nodes.explode('node_name')
df_question_nodes = df_question_nodes[
    df_question_nodes['node_name'].notna() & (df_question_nodes['node_name'] != '')
].drop(columns=['node_names'])

df_question_nodes = df_question_nodes.merge(
    df_dna_nodes[['node_id', 'node_name', 'source_topic',
                  'cognitive_operation', 'cognitive_demand', 'reasoning']],
    left_on=['node_name', 'topic'],
    right_on=['node_name', 'source_topic'],
    how='left'
).drop(columns=['source_topic', 'topic'])

df_question_nodes = df_question_nodes[
    ['question_id', 'node_id', 'node_name', 'cognitive_operation', 'cognitive_demand', 'reasoning']
]

print(f"\n✅ df_question_nodes : {df_question_nodes.shape}")
print(f"   Unmatched node_ids : {df_question_nodes['node_id'].isna().sum()}")


# ==========================================
# CELL 4: BRIDGE TABLE 2 — df_question_errors
# ==========================================
df_question_errors = df_errors.copy()
df_question_errors['error_name'] = df_question_errors['error_names'].apply(split_col)
df_question_errors = df_question_errors.explode('error_name')
df_question_errors = df_question_errors[
    df_question_errors['error_name'].notna() & (df_question_errors['error_name'] != '')
].drop(columns=['error_names'])

df_question_errors = df_question_errors.merge(
    df_error_tax[['error_id', 'error_name', 'topic', 'description', 'error_type']],
    on=['error_name', 'topic'],
    how='left'
).drop(columns=['topic'])

df_question_errors = df_question_errors[
    ['question_id', 'error_id', 'error_name', 'error_type', 'description']
]

print(f"\n✅ df_question_errors : {df_question_errors.shape}")
print(f"   Unmatched error_ids : {df_question_errors['error_id'].isna().sum()}")


# ==========================================
# CELL 5: EXPORT TO GOOGLE DRIVE
# ==========================================
# 1. Define the path to your mounted Google Drive folder
drive_path = '/content/drive/MyDrive/LECO/'

# 2. Export the DataFrames to CSV
df_master.to_csv(drive_path + 'df_master.csv', index=False)
df_question_nodes.to_csv(drive_path + 'df_question_nodes.csv', index=False)
df_question_errors.to_csv(drive_path + 'df_question_errors.csv', index=False)

print(f"\n✅ All files successfully saved to: {drive_path}")


✅ df_master : (4481, 13)
   Columns: ['question_id', 'topic', 'subject', 'year', 'shift', 'question_type', 'question_text', 'is_exemplar', 'exemplar_id', 'text_length', 'correct_answer', 'solution_steps', 'wrong_options']

✅ df_question_nodes : (5150, 6)
   Unmatched node_ids : 0

✅ df_question_errors : (5522, 5)
   Unmatched error_ids : 0

✅ All files successfully saved to: /content/drive/MyDrive/LECO/


In [ ]:
import pandas as pd
import re

# ── STEP 1: Fix topic capitalisation in source data ───────────────────────────
df_concepts['topic'] = df_concepts['topic'].str.replace('3D geometry', '3D Geometry', regex=False)

# ── STEP 2: Delimiter and split ───────────────────────────────────────────────
def split_col(val):
    if pd.isna(val) or str(val).strip() in ['', '[]', 'nan']:
        return []
    parts = [x.strip() for x in str(val).split(';') if x.strip()]
    return [p for p in parts if len(p) > 2 and re.search(r'[A-Za-z0-9]', p)]

# ── STEP 3: Fix map (one final version) ──────────────────────────────────────
concept_fix_map = {

    # Matrices
    'Tracking exponents across nested adj': 'Tracking exponents across nested adj, scalar, and power operations',
    'scalar': 'Tracking exponents across nested adj, scalar, and power operations',
    'and power operations': 'Tracking exponents across nested adj, scalar, and power operations',
    'Classifying parameter values for unique': 'Classifying parameter values for unique, infinite, or no solutions using determinant and rank conditions',
    'infinite': 'Classifying parameter values for unique, infinite, or no solutions using determinant and rank conditions',
    'or no solutions using determinant and rank conditions': 'Classifying parameter values for unique, infinite, or no solutions using determinant and rank conditions',
    'Discovering periodicity': 'Discovering periodicity, nilpotent structure, or closed-form patterns in matrix powers',
    'nilpotent structure': 'Discovering periodicity, nilpotent structure, or closed-form patterns in matrix powers',
    'or closed-form patterns in matrix powers': 'Discovering periodicity, nilpotent structure, or closed-form patterns in matrix powers',
    'Evaluating determinants using row and column operations on polynomial': 'Evaluating determinants using row and column operations on polynomial, factorial, or structured entries',
    'factorial': 'Evaluating determinants using row and column operations on polynomial, factorial, or structured entries',
    'or structured entries': 'Evaluating determinants using row and column operations on polynomial, factorial, or structured entries',

    # Functions
    'Computing fog gof for explicit polynomial/rational functions': 'Computing fog, gof for explicit polynomial/rational functions',
    'Involutory functions: f(f(x))=x finding parameter conditions': 'Involutory functions: f(f(x))=x, finding parameter conditions',
    'Absolute value compositions: f(|x|) |f(x)| and their combined behavior': 'Absolute value compositions: f(|x|), |f(x)|, and their combined behavior',
    'Domain with variable-base logarithms (base > 0 base ≠ 1 constraints)': 'Domain with variable-base logarithms (base > 0, base ≠ 1 constraints)',
    'max{f(x)': 'max{f(x), g(x)} and min{f(x), g(x)} as curve definitions',
    'g(x)} and min{f(x)': 'max{f(x), g(x)} and min{f(x), g(x)} as curve definitions',
    'g(x)} as curve definitions': 'max{f(x), g(x)} and min{f(x), g(x)} as curve definitions',
    'min{f(x)': 'max{f(x), g(x)} and min{f(x), g(x)} as curve definitions',
    'g(x)} — switching point analysis': 'max{f(x), g(x)} and min{f(x), g(x)} — switching point analysis',
    'g(x)} in area problems': 'max{f(x), g(x)} and min{f(x), g(x)} in area problems',

    # Probability
    "Computing P(A|B')": "Computing P(A|B'), P(A'∩B) and similar expressions from given event probabilities",
    "P(A'∩B) and similar expressions from given event probabilities": "Computing P(A|B'), P(A'∩B) and similar expressions from given event probabilities",
    'Counting favorable outcomes under divisibility': 'Counting favorable outcomes under divisibility, sum, or product constraints',
    'sum': 'Counting favorable outcomes under divisibility, sum, or product constraints',
    'or product constraints': 'Counting favorable outcomes under divisibility, sum, or product constraints',
    'Counting subsets satisfying divisibility': 'Counting subsets satisfying divisibility, coprimality, or modular arithmetic conditions',
    'coprimality': 'Counting subsets satisfying divisibility, coprimality, or modular arithmetic conditions',
    'or modular arithmetic conditions': 'Counting subsets satisfying divisibility, coprimality, or modular arithmetic conditions',
    'Counting triangles': 'Counting triangles, quadrilaterals, and polygons from distributed points',
    'quadrilaterals': 'Counting triangles, quadrilaterals, and polygons from distributed points',
    'and polygons from distributed points': 'Counting triangles, quadrilaterals, and polygons from distributed points',
    "De Morgan's laws": "Inclusion-exclusion, De Morgan's laws, and complement rules for compound events",
    'Inclusion-exclusion': "Inclusion-exclusion, De Morgan's laws, and complement rules for compound events",
    'and complement rules for compound events': "Inclusion-exclusion, De Morgan's laws, and complement rules for compound events",
    'Probability over randomly generated algebraic objects (matrices': 'Probability over randomly generated algebraic objects (matrices, quadratic equations, functions)',
    'quadratic equations': 'Probability over randomly generated algebraic objects (matrices, quadratic equations, functions)',
    'functions)': 'Probability over randomly generated algebraic objects (matrices, quadratic equations, functions)',
    'Divisibility': 'Divisibility, coprimality, and modular conditions as favorable-event filters',
    'and modular conditions as favorable-event filters': 'Divisibility, coprimality, and modular conditions as favorable-event filters',
    'Summing elements of filtered sets (HCF conditions': 'Summing elements of filtered sets (HCF conditions, residue classes)',
    'residue classes)': 'Summing elements of filtered sets (HCF conditions, residue classes)',

    # Sequences and Series
    'Parametric representation of AP terms (symmetric forms for 3': 'Parametric representation of AP terms (symmetric forms for 3, 4, 5 terms)',
    '5 terms)': 'Parametric representation of AP terms (symmetric forms for 3, 4, 5 terms)',
    'S_2n': 'Partial sum relationships (ratios of S_n, S_2n, S_3n) to infer common difference or first term',
    'S_3n) to infer common difference or first term': 'Partial sum relationships (ratios of S_n, S_2n, S_3n) to infer common difference or first term',
    'Partial sum relationships (ratios of S_n': 'Partial sum relationships (ratios of S_n, S_2n, S_3n) to infer common difference or first term',
    'Standard power-sum formulas (Σn': 'Standard power-sum formulas (Σn, Σn², Σn³) applied to split general terms',
    'Σn²': 'Standard power-sum formulas (Σn, Σn², Σn³) applied to split general terms',
    'Σn³) applied to split general terms': 'Standard power-sum formulas (Σn, Σn², Σn³) applied to split general terms',
    'Recognition of series as binomial expansion': 'Recognition of series as binomial expansion, exponential series, or their derivatives and integrals',
    'exponential series': 'Recognition of series as binomial expansion, exponential series, or their derivatives and integrals',
    'or their derivatives and integrals': 'Recognition of series as binomial expansion, exponential series, or their derivatives and integrals',

    # Limits
    'Standard limit equivalences (sinx/x → 1': 'Standard limit equivalences (sinx/x → 1, (e^x−1)/x → 1, log(1+x)/x → 1)',
    '(e^x−1)/x → 1': 'Standard limit equivalences (sinx/x → 1, (e^x−1)/x → 1, log(1+x)/x → 1)',
    'log(1+x)/x → 1)': 'Standard limit equivalences (sinx/x → 1, (e^x−1)/x → 1, log(1+x)/x → 1)',
    'Taylor/Maclaurin series expansion for sin': 'Taylor/Maclaurin series expansion for sin, cos, e^x, log(1+x), tan⁻¹(x)',
    'cos': 'Taylor/Maclaurin series expansion for sin, cos, e^x, log(1+x), tan⁻¹(x)',
    'e^x': 'Taylor/Maclaurin series expansion for sin, cos, e^x, log(1+x), tan⁻¹(x)',
    'log(1+x)': 'Taylor/Maclaurin series expansion for sin, cos, e^x, log(1+x), tan⁻¹(x)',
    'tan⁻¹(x)': 'Taylor/Maclaurin series expansion for sin, cos, e^x, log(1+x), tan⁻¹(x)',
    'Limits involving tan': 'Limits involving tan, cot, sec near their singularities',
    'cot': 'Limits involving tan, cot, sec near their singularities',
    'sec near their singularities': 'Limits involving tan, cot, sec near their singularities',

    # Differentiation
    'Derivatives of tan⁻¹': 'Derivatives of tan⁻¹, sin⁻¹, cot⁻¹ of algebraic arguments',
    'sin⁻¹': 'Derivatives of tan⁻¹, sin⁻¹, cot⁻¹ of algebraic arguments',
    'sin−1': 'Derivatives of tan⁻¹, sin⁻¹, cot⁻¹ of algebraic arguments',
    'cot⁻¹ of algebraic arguments': 'Derivatives of tan⁻¹, sin⁻¹, cot⁻¹ of algebraic arguments',
    'cot−1 of algebraic arguments': 'Derivatives of tan⁻¹, sin⁻¹, cot⁻¹ of algebraic arguments',

    # Definite Integration
    '∫_{0}^{nT} f(x)dx = n∫_0^T f(x)dx for periodic f': '∫_0^{nT} f(x)dx = n∫_0^T f(x)dx for periodic f',

    # Differential Equations
    'Exact grouping using total differentials (d(xy)': 'Exact grouping using total differentials (d(xy), d(y²/x))',
    'd(y²/x))': 'Exact grouping using total differentials (d(xy), d(y²/x))',
    'd(y2/x))': 'Exact grouping using total differentials (d(xy), d(y²/x))',

    # Quadratic Equations
    'Building a quadratic from indirect constraints (function values remainder conditions functional equations)': 'Building a quadratic from indirect constraints (function values, remainder conditions, functional equations)',
    'Conditions for both roots in a given interval exactly one root in an interval or roots of specified sign': 'Conditions for both roots in a given interval, exactly one root in an interval, or roots of specified sign',
    'Exponential-to-quadratic substitution (t = e^x t = a^x)': 'Exponential-to-quadratic substitution (t = e^x, t = a^x)',
    'Hidden quadratic in square-root floor-function or composite variable': 'Hidden quadratic in square-root, floor-function, or composite variable',

    # Vector Algebra
    'Angle between compound vectors constructed from sums differences or cross products': 'Angle between compound vectors constructed from sums, differences, or cross products',
    'Determining an unknown vector from simultaneous dot product cross product magnitude and coplanarity conditions': 'Determining an unknown vector from simultaneous dot product, cross product, magnitude, and coplanarity conditions',
    'Lagrange identity relating cross product magnitude dot product and vector magnitudes': 'Lagrange identity relating cross product magnitude, dot product, and vector magnitudes',
    'Scalar triple product properties: cyclic permutation linearity and [a×b b×c c×a] = [a b c]²': 'Scalar triple product properties: cyclic permutation, linearity, and [a×b, b×c, c×a] = [a b c]²',
    'Triangle properties from vertex position vectors (area centroid orthocentre circumcentre)': 'Triangle properties from vertex position vectors (area, centroid, orthocentre, circumcentre)',

    # 3D Geometry
    'Equation of line - vector and Cartesian form': 'Equation of line — vector and Cartesian form',

    # Parabola
    'Parametric representation (at² 2at)': 'Parametric representation (at², 2at)',
    'Standard equation y² = 4ax and its variants (x² = 4ay leftward/downward openings)': 'Standard equation y² = 4ax and its variants (x² = 4ay, leftward/downward openings)',
    'Tangent at a parametric point (ty = x + at²)': 'Tangent equation at a parametric point (ty = x + at²)',
    'Tangent to non-standard parabolas (y = x² y = ax² + bx + c)': 'Tangent to non-standard parabolas (y = x², y = ax² + bx + c)',
    'Area bounded by tangent parabola and axis': 'Area bounded by tangent, parabola, and axis',
    'Vertex axis latus rectum identification from equation': 'Vertex, axis, latus rectum identification from equation',

    # Ellipse — trailing commas
    'Completing the square for shifted ellipses,,': 'Completing the square for shifted ellipses',
    'Chord equation given midpoint (T = S₁),,': 'Chord equation given midpoint (T = S₁)',
    'Distance between foci,,': 'Distance between foci',
    'Distinguishing major axis orientation (horizontal vs vertical),': 'Distinguishing major axis orientation (horizontal vs vertical)',
    'Eccentricity computation from a, b,': 'Eccentricity computation from a, b',
    'Eccentricity computation from a, b,,': 'Eccentricity computation from a, b',
    'Ellipse with center not at origin,,': 'Ellipse with center not at origin',
    'Ellipse with center not at origin,,,': 'Ellipse with center not at origin',
    'Ellipse-circle intersection locus,': 'Ellipse-circle intersection locus',
    'Ellipse-hyperbola eccentricity relationship for confocal conics,': 'Ellipse-hyperbola eccentricity relationship for confocal conics',
    'Identifying a, b from general or standard equation,': 'Identifying a, b from general or standard equation',
    'Identifying a, b from general or standard equation,,': 'Identifying a, b from general or standard equation',
    'Inscribed triangle/quadrilateral area on ellipse,': 'Inscribed triangle/quadrilateral area on ellipse',
    'Length of chord with known midpoint,': 'Length of chord with known midpoint',
    'Length of chord with known midpoint,,': 'Length of chord with known midpoint',
    'Length of latus rectum (2b²/a),,': 'Length of latus rectum (2b²/a)',
    'Location of foci (c = ae),': 'Location of foci (c = ae)',
    'Location of foci (c = ae),,': 'Location of foci (c = ae)',
    'Locus of intersection of perpendicular tangents,,': 'Locus of intersection of perpendicular tangents',
    'Maximum inscribed triangle area,': 'Maximum inscribed triangle area',
    'Minimum area of triangle formed by tangent and axes,,': 'Minimum area of triangle formed by tangent and axes',
    'Parametric form — eccentric angle,,': 'Parametric form — eccentric angle',
    'Relationship b² = a²(1 − e²),': 'Relationship b² = a²(1 − e²)',
    'Relationship b² = a²(1 − e²),,': 'Relationship b² = a²(1 − e²)',
}

# ── STEP 4: Build bridge table ────────────────────────────────────────────────
df_concepts = df_concepts.copy()
df_concepts['concept_name'] = df_concepts['concept_names'].apply(split_col)
df_concepts = df_concepts.explode('concept_name')
df_concepts = df_concepts[
    df_concepts['concept_name'].notna() & (df_concepts['concept_name'] != '')
].drop(columns=['concept_names', 'file_name'], errors='ignore')

df_concepts['concept_name'] = (
    df_concepts['concept_name']
    .str.strip()
    .str.rstrip(',')
    .str.strip()
    .replace(concept_fix_map)
)

topic_override = {
    'Counting subsets satisfying divisibility, coprimality, or modular arithmetic conditions': 'Sets And Relations',
    'max{f(x), g(x)} and min{f(x), g(x)} as curve definitions': 'Area Under The Curves'

}
for concept, correct_topic in topic_override.items():
    df_concepts.loc[df_concepts['concept_name'] == concept, 'topic'] = correct_topic

# Drop concepts that don't exist in taxonomy
drop_list = [
    'Mean of first n natural numbers or A.P. set via linear transformation',
    'min{f(x), g(x)} and max{f(x), g(x)} as curve definitions',
    'max{f(x), g(x)} and min{f(x), g(x)} in area problems'
]
df_concepts = df_concepts[~df_concepts['concept_name'].isin(drop_list)]
# ── STEP 5: Merge ─────────────────────────────────────────────────────────────
df_question_concepts = df_concepts.merge(
    df_concepts_q[['concept_id', 'concept_name', 'topic']],
    on=['concept_name', 'topic'],
    how='left'
)[['question_id', 'concept_id', 'concept_name']]

# ── STEP 6: Report ────────────────────────────────────────────────────────────
unmatched = df_question_concepts[df_question_concepts['concept_id'].isna()]
print(f"Matched        : {df_question_concepts['concept_id'].notna().sum()}")
print(f"Still unmatched unique: {unmatched['concept_name'].nunique()}")
print(unmatched['concept_name'].value_counts())




drive_path = '/content/drive/MyDrive/LECO/'

# 2. Export the DataFrames to CSV
df_concepts.to_csv(drive_path + 'df_concepts.csv', index=False)


print(f"\n✅ All files successfully saved to: {drive_path}")

Matched        : 11784
Still unmatched unique: 0
Series([], Name: count, dtype: int64)

✅ All files successfully saved to: /content/drive/MyDrive/LECO/


In [ ]:
print(f"""
FINAL SCHEMA
─────────────────────────────────────────────────────
df_master      : {df_master.shape[0]} rows × {df_master.shape[1]} cols
df_question_nodes    : {df_question_nodes.shape[0]} rows × {df_question_nodes.shape[1]} cols
df_question_errors   : {df_question_errors.shape[0]} rows × {df_question_errors.shape[1]} cols
df_question_concepts : {df_question_concepts.shape[0]} rows × {df_question_concepts.shape[1]} cols
""")


FINAL SCHEMA
─────────────────────────────────────────────────────
df_master      : 4481 rows × 13 cols
df_question_nodes    : 5150 rows × 6 cols
df_question_errors   : 5522 rows × 5 cols
df_question_concepts : 11784 rows × 3 cols



In [ ]:
# ── 1. Nodes ──────────────────────────────────────────────────────────────────
print(f"Total unique nodes: {df_question_nodes['node_name'].nunique()}")
nodes_per_q = df_question_nodes.groupby('question_id')['node_name'].count()
print(f"Nodes per question (mean): {nodes_per_q.mean():.1f}")
print(f"Nodes per question (min/max): {nodes_per_q.min()} / {nodes_per_q.max()}")

# ── 2. Concepts ───────────────────────────────────────────────────────────────
print(f"\nTotal unique concepts: {df_question_concepts['concept_name'].nunique()}")
concepts_per_q = df_question_concepts.groupby('question_id')['concept_name'].count()
print(f"Concepts per question (mean): {concepts_per_q.mean():.1f}")
print(f"Concepts per question (min/max): {concepts_per_q.min()} / {concepts_per_q.max()}")

# ── 3. Errors ─────────────────────────────────────────────────────────────────
print(f"\nTotal unique errors: {df_question_errors['error_name'].nunique()}")
errors_per_q = df_question_errors.groupby('question_id')['error_name'].count()
print(f"Errors per question (mean): {errors_per_q.mean():.1f}")
print(f"Errors per question (min/max): {errors_per_q.min()} / {errors_per_q.max()}")
questions_with_errors = df_question_errors['question_id'].nunique()
print(f"Questions with no errors: {df_master['question_id'].nunique() - questions_with_errors}")

# ── 4. Node distribution ──────────────────────────────────────────────────────
node_counts = df_question_nodes['node_name'].value_counts()
print(f"\nTop 10 nodes by question count:")
print(node_counts.head(10))
print(f"\nBottom 10 nodes by question count:")
print(node_counts.tail(10))

# ── 5. Nodes per topic ────────────────────────────────────────────────────────
nodes_per_topic = (
    df_question_nodes
    .merge(df_master[['question_id', 'topic']], on='question_id', how='left')
    .groupby('topic')['node_name']
    .nunique()
    .sort_values(ascending=False)
)
print(f"\nUnique nodes per topic:")
print(nodes_per_topic)

Total unique nodes: 362
Nodes per question (mean): 1.1
Nodes per question (min/max): 1 / 3

Total unique concepts: 998
Concepts per question (mean): 2.6
Concepts per question (min/max): 1 / 11

Total unique errors: 286
Errors per question (mean): 1.2
Errors per question (min/max): 1 / 3
Questions with no errors: 16

Top 10 nodes by question count:
node_name
Integrating Factor Extraction from Non-Standard Groupings    113
Constraint-System Triangulation                               93
System-of-Equations Parameter Classification                  87
Foot-of-perpendicular extraction                              69
Symmetry Exploit                                              68
Piecewise Dissection                                          58
Monotonicity Interval Identification                          54
Disguise Unmasking                                            54
Indeterminate Form Resolution via Expansion                   53
Adjoint-Determinant Chain Unwinding                    